# Signals of Prestige in Film Discourse: A Transformer-Based Approach to Predicting Best Picture Winners (Model Development)

### By: Ed Hou, Si Qin Huang, Alejandro Mendez

---

Click here to [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-huang23/nlp-oscar-predictor/blob/main/model_development.ipynb)


<font color = 'orange'>**Preface:**

This notebook serves as the playground of the development of 3 models. Cells here are not designed to be the clean, final version, but rather demonstrate our thought process. For example, cells may illustrate swapping encoders and hyperparameter tuning for better results. For the final notebook, checkout final_oscar_predictor.ipynb on GitHub. </font>

In [ ]:
!pip install kagglehub sentence-transformers

In [ ]:
import kagglehub
import os
import glob
import json
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, AutoModel, AutoTokenizer
from typing import Optional

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

In [ ]:
# Mount drive (If manually uploaded data, mount drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Clone only the data folder from the repository (if not already present)
if not os.path.exists("data"):
    os.system("rm -rf /tmp/nlp-repo")
    ret = os.system("git clone --depth=1 https://github.com/s-huang23/nlp-oscar-predictor.git /tmp/nlp-repo")
    if ret == 0:
        os.system("cp -r /tmp/nlp-repo/data .")
        print("Data cloned successfully.")
    else:
        print("ERROR: git clone failed. Check that the repo is public and the URL is correct.")

print("data/ exists:", os.path.exists("data"))
print("data/raw/ exists:", os.path.exists("data/raw"))

Data cloned successfully.
data/ exists: True
data/raw/ exists: True


## Dataset

Load iMDb data from kaggle and web-scraped reviews from Metacritic

#### Metacritic data

In [ ]:
# Load metacritic dataset
# metacritic_df = pd.read_csv("/content/drive/MyDrive/metacritic_reviews.csv") # If data is uploaded to drive
metacritic_df = pd.read_csv("data/raw/metacritic_reviews.csv")
print(metacritic_df.shape)
metacritic_df.head()

(6643, 13)


,ceremony_year,release_year,film_title,winner,metacritic_slug,metacritic_url,critic_review_page,review_date,critic_score,publication,author,quote,full_review_url
0,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,The Hollywood Reporter,Kirk Honeycutt,"A fully believable, flesh-and-blood (albeit no...",http://www.hollywoodreporter.com/hr/film-revie...
1,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,Empire,Chris Hewitt (1),"It's been twelve years since ""Titanic,"" but th...",http://www.empireonline.com/reviews/reviewcomp...
2,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,90,Variety,Todd McCarthy,"Avatar is all-enveloping and transporting, wit...",http://www.variety.com/review/VE1117941773.htm...
3,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,Chicago Sun-Times,Roger Ebert,"Once again, [Cameron] has silenced the doubter...",http://rogerebert.suntimes.com/apps/pbcs.dll/a...
4,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,75,Chicago Tribune,Michael Phillips,The first 90 minutes of Avatar are pretty terr...,http://featuresblogs.chicagotribune.com/talkin...


#### Kaggle iMDb data

In [ ]:
# Load nominees dataset
# nominees_df = pd.read_csv("/content/drive/MyDrive/nominees.csv") # If data is uploaded to drive
nominees_df   = pd.read_csv("data/nominees.csv")
print(nominees_df.shape)
nominees_df.head()

(136, 5)


,ceremony_year,film_title,release_year,winner,metacritic_slug
0,2010,Avatar,2009,0,avatar
1,2010,The Blind Side,2009,0,the-blind-side
2,2010,District 9,2009,0,district-9
3,2010,An Education,2009,0,an-education
4,2010,The Hurt Locker,2009,1,the-hurt-locker


In [ ]:
# Download latest version of data from kaggle
path = kagglehub.dataset_download("ebiswas/imdb-review-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-review-dataset' dataset.
Path to dataset files: /kaggle/input/imdb-review-dataset


In [ ]:
# Making sure title discrepancies are fixed
title_mapping = {
    "Extremely Loud and Incredibly Close": "Extremely Loud & Incredibly Close",
    "Les Miserables": "Les MisÃ©rables",
    "Once Upon a Time ... in Hollywood" : "Once Upon a Time... In Hollywood",
    "Hell or High Water" : "Hell or High Water (II)",
    "Boyhood" : "Boyhood (I)",
    "Arrival" : "Arrival (II)",
    "Get Out" : "Get Out (I)",
    "Moonlight" : "Moonlight (I)",
    "Room" : "Room (I)",
    "Spotlight" : "Spotlight (I)",
    "The Artist" : "The Artist (I)",
    "Vice" : "Vice (I)"
}

# Mapping title fixes
nominees_df["film_title"] = nominees_df["film_title"].replace(title_mapping)
metacritic_df["film_title"] = metacritic_df["film_title"].replace(title_mapping)

# Verify the fix
#print(nominees_df[nominees_df["film_title"].isin(title_mapping.values())][["film_title", "ceremony_year"]])

In [ ]:
# print(os.listdir(path))

In [ ]:
# Combine all parts of json
all_files = glob.glob(os.path.join(path, "part-*.json"))

records = []
for f in all_files:
    with open(f) as fh:
        records.extend(json.load(fh))

# Convert to dataframe
reviews_df = pd.DataFrame(records)
print(reviews_df.shape)
reviews_df.head()

(5571499, 9)


,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful
0,rw1133942,OriginalMovieBuff21,Kill Bill: Vol. 2 (2004),8,Good follow up that answers all the questions,24 July 2005,0,"After seeing Tarantino's Kill Bill Vol: 1, I g...","[0, 1]"
1,rw1133943,sentra14,Journey to the Unknown (1968– ),None,Excellent series,24 July 2005,0,"I have the entire series on video, taped mostl...","[11, 11]"
2,rw1133946,GreenwheelFan2002,The Island (2005),9,"Not just about action, but about survival...",24 July 2005,0,Once again the critics prove themselves as mor...,"[2, 5]"
3,rw1133948,itsascreambaby,Win a Date with Tad Hamilton! (2004),3,Falls under the category: seen it a million ti...,24 July 2005,0,This IS a film that has been done too many tim...,"[2, 3]"
4,rw1133949,OriginalMovieBuff21,Saturday Night Live: The Best of Chris Farley ...,10,"Before Tommy Boy and Black Sheep, there was Sa...",24 July 2005,0,Chris Farley is one of my favorite comedians a...,"[4, 4]"


In [ ]:
# Extract title and year from "Movie Title (YEAR)" format
reviews_df["release_year"] = (
    reviews_df["movie"]
    .str.extract(r'\((\d{4})')   # grab the first 4-digit year after '('
    .astype(float)
    .astype("Int64")
)

reviews_df["movie"] = (
    reviews_df["movie"]
    .str.replace(r'\s*\(\d{4}[^)]*\)', '', regex=True)  # remove anything from (YEAR...  to )
    .str.strip()
)

print(reviews_df[["movie", "release_year"]].head(10))

                                           movie  release_year
0                              Kill Bill: Vol. 2          2004
1                         Journey to the Unknown          1968
2                                     The Island          2005
3                  Win a Date with Tad Hamilton!          2004
4  Saturday Night Live: The Best of Chris Farley          2000
5                                    Outlaw Star          1998
6                                    The Aviator          2004
7      Star Wars: Episode I - The Phantom Menace          1999
8                          The Amityville Horror          2005
9                                  Flying Tigers          1942


In [ ]:
# Filter and merge using both columns
filtered_reviews = reviews_df[
    reviews_df["movie"].isin(nominees_df["film_title"]) &
    reviews_df["release_year"].isin(nominees_df["release_year"])
].copy()

print(f"Total reviews matched: {len(filtered_reviews)}")
print(f"Unique films matched : {filtered_reviews['movie'].nunique()} / {len(nominees_df)}")

imdb_df = filtered_reviews.merge(
    nominees_df[["film_title", "release_year", "ceremony_year", "winner"]],
    left_on=["movie", "release_year"],
    right_on=["film_title", "release_year"],
    how="left"
)

# Drop values with NAs that don't match up
imdb_df = imdb_df.dropna(subset=["winner"])
# Fix winner and ceremony_year column formatting
imdb_df["winner"] = imdb_df["winner"].astype(int)
imdb_df["ceremony_year"] = imdb_df["ceremony_year"].astype(int)

print(imdb_df.shape)
imdb_df.head()

Total reviews matched: 100154
Unique films matched : 107 / 136
(99723, 13)


,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful,release_year,film_title,ceremony_year,winner
0,rw2073111,jdesando,Up,None,A Summer Upper,28 May 2009,0,"""The thirst for adventure is the vent which De...","[7, 15]",2009,Up,2010,0
1,rw2073175,nirmal_vijay,Up,10,Up you go.......,28 May 2009,0,10 in a row for Pixar. Full of depth in imagin...,"[3, 9]",2009,Up,2010,0
2,rw2073244,TheWylde,Up,8,Pixar's best since Toy Story 2,29 May 2009,0,"Admittedly, I don't really like children's mov...","[1, 11]",2009,Up,2010,0
3,rw2073354,aleexx_1920,Up,10,up is pixar's magic,29 May 2009,0,OK I was never interested in seeing this film ...,"[2, 8]",2009,Up,2010,0
4,rw2073435,gro,Up,4,Up is Down,29 May 2009,1,"In the past couple months I've seen Coraline, ...","[13, 42]",2009,Up,2010,0


In [ ]:
# Drop unneeded columns
try:
  imdb_df = imdb_df.drop(columns=["movie", "review_summary", "spoiler_tag", "helpful"])
except:
  print("IMDB_df cols already dropped")

print(imdb_df.columns.tolist())
print(imdb_df["winner"].value_counts())
imdb_df.head()

#Need to move this to earlier preprocessing
imdb_df["rating"] = pd.to_numeric(imdb_df["rating"], errors="coerce")

#Les Mix fix STILL not working
imdb_df["film_title"] = imdb_df["film_title"].str.encode("utf-8", errors="ignore").str.decode("utf-8")
metacritic_df["film_title"] = metacritic_df["film_title"].str.encode("utf-8", errors="ignore").str.decode("utf-8")

# Then re-apply the title mapping
imdb_df["film_title"] = imdb_df["film_title"].replace(title_mapping)
metacritic_df["film_title"] = metacritic_df["film_title"].replace(title_mapping)

#This is bc still have old years
nominees_df_filtered = nominees_df[
    (nominees_df["ceremony_year"] >= 2012) &
    (nominees_df["ceremony_year"] <= 2020)
].copy()

['review_id', 'reviewer', 'rating', 'review_date', 'review_detail', 'release_year', 'film_title', 'ceremony_year', 'winner']
winner
0    88411
1    11312
Name: count, dtype: int64


## Pre-processing

Implemented steps:
  - Filter to ceremony years 2012-2020
  - Remove all reviews after ceremony date
  - Remove short or empty reviews and duplicate review text
  - Add per-film review counts and inverse review weights for volume normalization
  - Clean review text into `clean_text`

In [ ]:
# Preprocess IMDb and Metacritic reviews.
# Core window: review_date < ceremony_date.
import re

START_YEAR = 2012
END_YEAR = 2020
MIN_TEXT_CHARS = 20

# windows_path = "/content/drive/MyDrive/oscar_windows.csv"
oscar_windows = pd.read_csv("data/oscar_windows.csv", parse_dates=["nomination_date", "ceremony_date"])


def clean_text(value):
    """Normalize review text while preserving the words used by reviewers."""
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# Sanity check: emoji can carry sentiment, so preprocessing should keep them.
assert clean_text("Loved it 😭!") == "Loved it 😭!"


def summarize_reviews(label, df):
    film_count = df[["ceremony_year", "film_title"]].drop_duplicates().shape[0]
    years = sorted(df["ceremony_year"].dropna().astype(int).unique().tolist()) if len(df) else []
    print(f"{label:<36} rows={len(df):>6} films={film_count:>3} years={years}")


def preprocess_reviews(df, source_name, text_col):
    """Apply project preprocessing to one review source."""
    print(f"\n{source_name.upper()} preprocessing")
    summarize_reviews("raw", df)

    working = df[
        (df["ceremony_year"] >= START_YEAR) &
        (df["ceremony_year"] <= END_YEAR)
    ].copy()
    summarize_reviews("after ceremony-year filter", working)

    working = working.merge(oscar_windows, on="ceremony_year", how="left", validate="many_to_one")
    working["review_date_parsed"] = pd.to_datetime(working["review_date"], errors="coerce")

    invalid_dates = working["review_date_parsed"].isna().sum()
    if invalid_dates:
        print(f"dropped invalid review dates: {invalid_dates}")

    in_window = (
        working["review_date_parsed"].notna()
        & (working["review_date_parsed"] < working["ceremony_date"])
    )
    working = working[in_window].copy()
    summarize_reviews("after ceremony-date filter", working)

    working["clean_text"] = working[text_col].map(clean_text)
    before = len(working)
    working = working[working["clean_text"].str.len() >= MIN_TEXT_CHARS].copy()
    print(f"dropped short/empty reviews: {before - len(working)}")

    dedupe_cols = ["ceremony_year", "film_title", "clean_text"]
    if source_name == "imdb" and "reviewer" in working.columns:
        dedupe_cols.insert(2, "reviewer")
    if source_name == "metacritic" and "publication" in working.columns:
        dedupe_cols.insert(2, "publication")

    before = len(working)
    working = working.drop_duplicates(subset=dedupe_cols).copy()
    print(f"dropped duplicate reviews: {before - len(working)}")

    working["film_review_count"] = working.groupby(
        ["ceremony_year", "film_title"]
    )["clean_text"].transform("size")
    working["film_review_weight"] = 1.0 / working["film_review_count"]

    summarize_reviews("final", working)
    return working

imdb_df = preprocess_reviews(imdb_df, "imdb", "review_detail")
metacritic_df = preprocess_reviews(metacritic_df, "metacritic", "quote")


IMDB preprocessing
raw                                  rows= 99723 films= 98 years=[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
after ceremony-year filter           rows= 82558 films= 77 years=[2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
after ceremony-date filter           rows= 42142 films= 77 years=[2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
dropped short/empty reviews: 2
dropped duplicate reviews: 13
final                                rows= 42127 films= 77 years=[2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

METACRITIC preprocessing
raw                                  rows=  6643 films=136 years=[2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
after ceremony-year filter           rows=  3787 films= 78 years=[2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
after ceremony-date filter           rows=  3730 films= 78 years=[2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

In [ ]:
meta_films = set(metacritic_df["film_title"].unique())
imdb_films = set(imdb_df["film_title"].unique())

print("In Metacritic but NOT in IMDb:")
print(meta_films - imdb_films)

In Metacritic but NOT in IMDb:
{'Les MisÃ©rables'}


## Model 1

**Transformer with Cross-Attention between IMDb user + Metacritic critic reviews encoded on shared RoBERTa Base**

**Hypothesis:** domain-separated encoding with bidirectional
cross-attention between critic and audience discourse captures
complementary signals that improve Best Picture prediction

**Features**
- RoBERTa base: improved version of BERT with 10x training data and improved hyperparameter tuning & dynamic masking
- Segmented feature space of IMDb user reviews and Metacritic reviews
- Both review streams pass through 2 self-attention layers, allowing each review to update its representation by attending to all other reviews of the same film
- Multi-head
- Cross-attention: explore relation between IMDb user reviews and Metacritic critic reviews
- Fusion layers: difference and product instead of simple concatenation

**Initial Model 1 (RoBERTa base) class:**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class OscarPredictorAttention(nn.Module):
    """
    Dual-stream model with real multi-head self-attention and cross-attention.

    Pipeline per film:
      1. BERT encodes each review → token-level representations
      2. Mean pool tokens → one vector per review
      3. Self-attention across reviews within each stream
      4. Cross-attention between streams (critic ↔ IMDb)
      5. Pooling → one vector per stream
      6. Fusion → single film logit
    """
    def __init__(
        self,
        model_name="roberta-base",
        hidden_dim=768,
        num_heads=8,
        num_self_attn_layers=2,
        dropout=0.3
    ):
        super(OscarPredictorAttention, self).__init__()

        self.hidden_dim = hidden_dim

        # Shared BERT-family encoder
        self.encoder = AutoModel.from_pretrained(model_name)

        # Freeze BERT — only train attention + fusion given small dataset
        for param in self.encoder.parameters():
            param.requires_grad = False

        # ── Self-attention over reviews within each space (critic or user) ────────────────────
        # Stacked transformer encoder layers — each review attends to all
        # other reviews in the same stream
        critic_self_attn_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,        # 8 heads looking at different aspects like sentiment or emotional weight, or film aspects like acting, cinematography
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",      #improved ReLU that allows SOME small gradient to flow through (not zero if negative)
            batch_first=True        # [batch, seq, dim] not [seq, batch, dim]
        )
        # num_self_attn_layers=2 means two rounds of self-attention
        self.critic_self_attn = nn.TransformerEncoder(
            critic_self_attn_layer,
            num_layers=num_self_attn_layers
        )

        imdb_self_attn_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.imdb_self_attn = nn.TransformerEncoder(
            imdb_self_attn_layer,
            num_layers=num_self_attn_layers
        )

        # ── Cross-attention between streams ───────────────────────────────────
        # Critics querying into IMDb
        # Q=critic, K=IMDb, V=IMDb
        # "for each critic review, what IMDb reviews are most relevant?"
        self.cross_attn_critic_to_imdb = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # IMDb querying into critics
        # Q=IMDb, K=critic, V=critic
        # "for each IMDb review, what critic reviews are most relevant?"
        self.cross_attn_imdb_to_critic = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # Layer norms — stabilize training after each attention operation
        self.norm_critic_self  = nn.LayerNorm(hidden_dim)
        self.norm_imdb_self    = nn.LayerNorm(hidden_dim)
        self.norm_critic_cross = nn.LayerNorm(hidden_dim)
        self.norm_imdb_cross   = nn.LayerNorm(hidden_dim)

        # FUSION
        # diff and product give the model explicit relationship signals
        # between the two streams rather than just a simple concatenation
        fusion_input_dim = hidden_dim * 4  # critic + imdb + diff + product

        self.scorer = nn.Sequential(
            nn.Linear(fusion_input_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
            # no sigmoid — softmax applied across cohort year at training time
        )

    def encode_reviews(self, input_ids, attention_mask):
        """
        Step 1: BERT encodes each review - each film has many reviews each with many tokens
        -> mean pool tokens -> one vector per review.

        input_ids      : [batch, num_reviews, seq_len]
        attention_mask : [batch, num_reviews, seq_len]
        returns        : [batch, num_reviews, hidden_dim]

        NOTE: encoder is NOT universal but model-dependent
        """
        batch_size, num_reviews, seq_len = input_ids.shape

        # Flatten for parallel BERT encoding
        ids_flat  = input_ids.reshape(-1, seq_len)
        mask_flat = attention_mask.reshape(-1, seq_len) #padding 0's for shorter reviews

        with torch.no_grad():  # frozen — skip gradient computation
          outputs = self.encoder(input_ids=ids_flat, attention_mask=mask_flat)

        # Mean pool over token dimension
        hidden   = outputs.last_hidden_state              # [batch*num_reviews, seq_len, 768]
        mask_exp = mask_flat.unsqueeze(-1).float()
        pooled   = (hidden * mask_exp).sum(dim=1) / \
                   mask_exp.sum(dim=1).clamp(min=1e-8)   # [batch*num_reviews, 768]

        # Reshape back — now have one vector per review
        return pooled.reshape(batch_size, num_reviews, -1) # [batch, num_reviews, 768]

    def mean_pool_reviews(self, x):
        """Collapse [batch, num_reviews, 768] → [batch, 768]"""
        return x.mean(dim=1)

    def forward(
        self,
        critic_input_ids,    critic_attention_mask,
        imdb_input_ids,      imdb_attention_mask,
    ):
        # 1. BERT encoding
        # [batch, num_reviews, 768] — one vector per review
        h_critic = self.encode_reviews(critic_input_ids, critic_attention_mask)
        h_imdb   = self.encode_reviews(imdb_input_ids,   imdb_attention_mask)

        # 2. Self-attention within each stream
        # Each critic review now attends to all other critic reviews
        # TransformerEncoder handles the padding mask to ignore padded reviews
        critic_pad_mask = (critic_attention_mask.sum(dim=-1) == 0)  # [batch, num_reviews]
        imdb_pad_mask   = (imdb_attention_mask.sum(dim=-1)   == 0)

        h_critic_self = self.critic_self_attn(
            h_critic,
            src_key_padding_mask=critic_pad_mask
        )                                                  # [batch, num_reviews, 768]
        h_critic = self.norm_critic_self(h_critic + h_critic_self)  # residual connection

        h_imdb_self = self.imdb_self_attn(
            h_imdb,
            src_key_padding_mask=imdb_pad_mask
        )
        h_imdb = self.norm_imdb_self(h_imdb + h_imdb_self)

        # 3. Cross-attention between streams
        # Critics query into IMDb
        # Each critic review asks: "which IMDb reviews are most relevant to me?"
        h_critic_cross, _ = self.cross_attn_critic_to_imdb(
            query=h_critic,                                # Q: critic reviews
            key=h_imdb,                                    # K: IMDb reviews
            value=h_imdb,                                  # V: IMDb reviews
            key_padding_mask=imdb_pad_mask
        )
        h_critic = self.norm_critic_cross(h_critic + h_critic_cross)  # residual

        # IMDb queries into critics
        h_imdb_cross, _ = self.cross_attn_imdb_to_critic(
            query=h_imdb,
            key=h_critic,
            value=h_critic,
            key_padding_mask=critic_pad_mask
        )
        h_imdb = self.norm_imdb_cross(h_imdb + h_imdb_cross)

        # 4. Pool to one vector per stream
        c_critic = self.mean_pool_reviews(h_critic)        # [batch, 768]
        c_imdb   = self.mean_pool_reviews(h_imdb)          # [batch, 768]

        # 5. Fusion
        diff    = c_critic - c_imdb                        # disagreement signal
        product = c_critic * c_imdb                        # agreement signal

        fused = torch.cat([c_critic, c_imdb, diff, product], dim=-1)  # [batch, 3072]

        return self.scorer(fused)                          # [batch, 1] raw logit

**DataLoaders, Encoding, + Supporting Infrastructure for Model 1 (RoBERTa base)**

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# ── DATASET ───────────────────────────────────────────────────────────────────
class OscarCohortDataset(Dataset):
    def __init__(self, nominees_df, metacritic_df, imdb_df,
                 tokenizer, max_critic_reviews=40, max_imdb_reviews=50,
                 max_len=128, seed=42):

        self.tokenizer          = tokenizer
        self.max_critic_reviews = max_critic_reviews
        self.max_imdb_reviews   = max_imdb_reviews
        self.max_len            = max_len
        self.seed               = seed
        self.cohorts            = []

        for year, year_nominees in nominees_df.groupby("ceremony_year"):
            films      = year_nominees["film_title"].tolist()
            winner_row = year_nominees[year_nominees["winner"] == 1]

            if winner_row.empty:
                print(f"  Warning: no winner found for {year}, skipping")
                continue

            winner_title = winner_row["film_title"].iloc[0]

            if winner_title not in films:
                print(f"  Warning: winner {winner_title} not in films for {year}, skipping")
                continue

            winner_idx = films.index(winner_title)

            critic_reviews_per_film = []
            imdb_reviews_per_film   = []

            for film in films:
                # Metacritic — clean_text column
                critic_pool = metacritic_df[
                    (metacritic_df["film_title"]    == film) &
                    (metacritic_df["ceremony_year"] == year)
                ]["clean_text"].dropna().tolist()

                # IMDb — clean_text column, stratified sample
                imdb_pool_df = imdb_df[
                    (imdb_df["film_title"]    == film) &
                    (imdb_df["ceremony_year"] == year)
                ].dropna(subset=["clean_text"])

                if len(imdb_pool_df) > self.max_imdb_reviews and "rating" in imdb_pool_df.columns:
                    imdb_pool = self._stratified_sample(
                        imdb_pool_df, n=self.max_imdb_reviews
                    )["clean_text"].tolist()
                else:
                    imdb_pool = imdb_pool_df["clean_text"].tolist()[:self.max_imdb_reviews]

                # Fallbacks for missing reviews
                if not critic_pool:
                    print(f"  Warning: no critic reviews for {film} ({year})")
                    critic_pool = ["no critic reviews available"]

                if not imdb_pool:
                    print(f"  Warning: no IMDb reviews for {film} ({year})")
                    imdb_pool = ["no user reviews available"]

                critic_reviews_per_film.append(critic_pool)
                imdb_reviews_per_film.append(imdb_pool)

            self.cohorts.append({
                "year":           year,
                "films":          films,
                "critic_reviews": critic_reviews_per_film,
                "imdb_reviews":   imdb_reviews_per_film,
                "winner_idx":     winner_idx,
            })

        print(f"Dataset built: {len(self.cohorts)} cohorts")

    def _stratified_sample(self, df, n):
        df = df.copy()
        df["bucket"] = pd.cut(
            df["rating"],
            bins=[0, 4, 7, 10],
            labels=["negative", "mixed", "positive"],
            include_lowest=True
        )
        frames = []
        for bucket, prop in df["bucket"].value_counts(normalize=True).items():
            k          = max(1, round(prop * n))
            bucket_df  = df[df["bucket"] == bucket]
            k          = min(k, len(bucket_df))
            frames.append(bucket_df.sample(n=k, random_state=self.seed))
        return pd.concat(frames).head(n)

    def _tokenize(self, reviews, max_reviews):
        reviews = [r for r in reviews if isinstance(r, str) and len(r.strip()) > 0]
        reviews = reviews[:max_reviews]
        if not reviews:
            reviews = [""]

        encoded = self.tokenizer(
            reviews,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        ids  = encoded["input_ids"]
        mask = encoded["attention_mask"]

        # Pad up to max_reviews if fewer reviews exist
        n = ids.shape[0]
        if n < max_reviews:
            pad_ids  = torch.zeros(max_reviews - n, self.max_len, dtype=torch.long)
            pad_mask = torch.zeros(max_reviews - n, self.max_len, dtype=torch.long)
            ids  = torch.cat([ids,  pad_ids],  dim=0)
            mask = torch.cat([mask, pad_mask], dim=0)

        return ids, mask

    def __len__(self):
        return len(self.cohorts)

    def __getitem__(self, idx):
        cohort  = self.cohorts[idx]
        n_films = len(cohort["films"])

        critic_ids_list, critic_mask_list = [], []
        imdb_ids_list,   imdb_mask_list   = [], []

        for i in range(n_films):
            c_ids, c_mask = self._tokenize(cohort["critic_reviews"][i], self.max_critic_reviews)
            i_ids, i_mask = self._tokenize(cohort["imdb_reviews"][i],   self.max_imdb_reviews)

            critic_ids_list.append(c_ids)
            critic_mask_list.append(c_mask)
            imdb_ids_list.append(i_ids)
            imdb_mask_list.append(i_mask)

        return {
            "year":                  cohort["year"],
            "films":                 cohort["films"],
            "critic_input_ids":      torch.stack(critic_ids_list),    # [n_films, 40, 128]
            "critic_attention_mask": torch.stack(critic_mask_list),
            "imdb_input_ids":        torch.stack(imdb_ids_list),      # [n_films, 50, 128]
            "imdb_attention_mask":   torch.stack(imdb_mask_list),
            "winner_idx":            torch.tensor(cohort["winner_idx"], dtype=torch.long),
        }


# ── TRAIN / EVAL STEPS ────────────────────────────────────────────────────────
def train_step(model, optimizer, batch, device):
    model.train()
    optimizer.zero_grad()

    critic_ids  = batch["critic_input_ids"].squeeze(0).to(device)
    critic_mask = batch["critic_attention_mask"].squeeze(0).to(device)
    imdb_ids    = batch["imdb_input_ids"].squeeze(0).to(device)
    imdb_mask   = batch["imdb_attention_mask"].squeeze(0).to(device)
    winner_idx  = batch["winner_idx"].squeeze().to(device)  # flatten to scalar

    logits = model(critic_ids, critic_mask, imdb_ids, imdb_mask)

    loss = nn.CrossEntropyLoss()(
        logits.squeeze(-1).unsqueeze(0),   # [1, n_films]
        winner_idx.unsqueeze(0)            # [1]
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    probs     = torch.softmax(logits.squeeze(-1), dim=0)
    predicted = probs.argmax().item()
    correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


def eval_step(model, batch, device):
    model.eval()
    with torch.no_grad():
        critic_ids  = batch["critic_input_ids"].squeeze(0).to(device)
        critic_mask = batch["critic_attention_mask"].squeeze(0).to(device)
        imdb_ids    = batch["imdb_input_ids"].squeeze(0).to(device)
        imdb_mask   = batch["imdb_attention_mask"].squeeze(0).to(device)
        winner_idx  = batch["winner_idx"].squeeze().to(device)  # flatten to scalar

        logits = model(critic_ids, critic_mask, imdb_ids, imdb_mask)

        loss = nn.CrossEntropyLoss()(
            logits.squeeze(-1).unsqueeze(0),
            winner_idx.unsqueeze(0)
        )

        probs     = torch.softmax(logits.squeeze(-1), dim=0)
        predicted = probs.argmax().item()
        correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()

Device: cuda
Running smoke test...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

  Epoch 01 | loss 2.2844 | train acc 1/8
  Epoch 02 | loss 2.1449 | train acc 0/8
  Epoch 03 | loss 2.3867 | train acc 0/8
  Epoch 04 | loss 2.1060 | train acc 1/8
  Epoch 05 | loss 2.0512 | train acc 1/8
  Epoch 06 | loss 2.2733 | train acc 0/8
  Epoch 07 | loss 2.1226 | train acc 0/8
  Epoch 08 | loss 2.1820 | train acc 3/8
  Epoch 09 | loss 2.2375 | train acc 0/8
  Epoch 10 | loss 2.2174 | train acc 0/8

  Results for 2014:
    American Hustle                               0.111
    Captain Phillips                              0.112
    Dallas Buyers Club                            0.111
    Gravity                                       0.111
    Her                                           0.112 ← predicted
    Nebraska                                      0.111
    Philomena                                     0.111
    12 Years a Slave                              0.111 ← winner
    The Wolf of Wall Street                       0.111

Fold: test year = 2015
Dataset built: 8 coh

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 01 | loss 2.3642 | train acc 0/8


KeyboardInterrupt: 

**Model 1 (RoBERTa base) Training**

In [ ]:
# ── SANITY CHECK ──────────────────────────────────────────────────────────────
def smoke_test(nominees_df, metacritic_df, imdb_df, device="cpu"):
    """
    Run one forward pass on one cohort before full training.
    Catches shape errors, missing columns, and device issues early.
    """
    print("Running smoke test...")
    tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    dataset   = OscarCohortDataset(
        nominees_df, metacritic_df, imdb_df, tokenizer,
        max_critic_reviews=5, max_imdb_reviews=5, max_len=64  # tiny for speed
    )

    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    batch  = next(iter(loader))

    print(f"  Year: {batch['year']}")
    print(f"  Films: {batch['films']}")
    print(f"  Winner idx: {batch['winner_idx'].item()}")
    print(f"  critic_input_ids shape: {batch['critic_input_ids'].shape}")
    print(f"  imdb_input_ids shape:   {batch['imdb_input_ids'].shape}")

    model  = OscarPredictorAttention().to(device)
    logits = model(
        batch["critic_input_ids"].squeeze(0).to(device),
        batch["critic_attention_mask"].squeeze(0).to(device),
        batch["imdb_input_ids"].squeeze(0).to(device),
        batch["imdb_attention_mask"].squeeze(0).to(device),
    )

    print(f"  Output logits shape: {logits.shape}")   # should be [n_films, 1]
    print(f"  Logits: {logits.squeeze(-1).tolist()}")
    print("Smoke test passed.")
    return True


# ── REAL TRAINING ─────────────────────────────────────────────────────────────
def run_loocv(nominees_df, metacritic_df, imdb_df,
              num_epochs=10, lr=1e-4, regularizer=0.01,  # fixed missing comma
              max_critic_reviews=40, max_imdb_reviews=50,
              max_len=128, patience=5, device="cpu"):

    tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    all_years = sorted(nominees_df["ceremony_year"].unique())
    results   = []

    for test_year in all_years:
        print(f"\n{'='*56}")
        print(f"Fold: test year = {test_year}")
        print(f"{'='*56}")

        # Split by year
        train_nom  = nominees_df[nominees_df["ceremony_year"] != test_year]
        test_nom   = nominees_df[nominees_df["ceremony_year"] == test_year]
        train_meta = metacritic_df[metacritic_df["ceremony_year"] != test_year]
        test_meta  = metacritic_df[metacritic_df["ceremony_year"] == test_year]
        train_imdb = imdb_df[imdb_df["ceremony_year"] != test_year]
        test_imdb  = imdb_df[imdb_df["ceremony_year"] == test_year]

        # Build datasets
        train_dataset = OscarCohortDataset(
            train_nom, train_meta, train_imdb, tokenizer,
            max_critic_reviews=max_critic_reviews,
            max_imdb_reviews=max_imdb_reviews,
            max_len=max_len
        )
        test_dataset = OscarCohortDataset(
            test_nom, test_meta, test_imdb, tokenizer,
            max_critic_reviews=max_critic_reviews,
            max_imdb_reviews=max_imdb_reviews,
            max_len=max_len
        )

        train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
        test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)

        # Fresh model per fold
        model     = OscarPredictorAttention().to(device)
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=regularizer
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs
        )

        # Early stopping
        best_loss  = float("inf")
        no_improve = 0
        best_state = None

        # Training epochs
        for epoch in range(num_epochs):
            losses, corrects = [], []

            for batch in train_loader:
                loss, correct, _ = train_step(model, optimizer, batch, device)
                losses.append(loss)
                corrects.append(correct)

            scheduler.step()
            mean_loss = np.mean(losses)
            print(f"  Epoch {epoch+1:02d} | loss {mean_loss:.4f} | "
                  f"train acc {sum(corrects)}/{len(corrects)}")

            # Early stopping — save best weights
            if mean_loss < best_loss - 1e-4:
                best_loss  = mean_loss
                no_improve = 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"  Early stopping at epoch {epoch+1} "
                          f"(best loss {best_loss:.4f})")
                    break

        # Restore best weights before evaluation
        if best_state is not None:
            model.load_state_dict(best_state)

        # Evaluate on held-out year
        print(f"\n  Results for {test_year}:")
        for batch in test_loader:
            loss, correct, probs = eval_step(model, batch, device)
            films      = batch["films"]
            winner_idx = batch["winner_idx"].item()
            pred_idx   = probs.index(max(probs))

            for i, (film, prob) in enumerate(zip(films, probs)):  # fixed indent
                winner_marker = " ← winner"    if i == winner_idx else ""
                pred_marker   = " ← predicted" if i == pred_idx   else ""
                print(f"    {film[0]:45s} {prob:.3f}{winner_marker}{pred_marker}")

            # Compute winner rank for MRR
            ranked_films = [f for f, p in sorted(
                zip([fi[0] for fi in films], probs),
                key=lambda x: x[1], reverse=True
            )]
            winner_rank = ranked_films.index(films[winner_idx][0]) + 1

            results.append({
                "test_year":   test_year,
                "correct":     correct,
                "winner":      films[winner_idx][0],
                "predicted":   films[pred_idx][0],
                "probs":       probs,
                "films":       [f[0] for f in films],
                "winner_rank": winner_rank,
            })

    # Final summary
    n    = len(results)
    top1 = sum(r["correct"] for r in results)
    top3 = sum(1 for r in results
               if r["winner"] in [f for f, p in sorted(
                   zip(r["films"], r["probs"]),
                   key=lambda x: x[1], reverse=True)][:3])
    mrr  = np.mean([1.0 / r["winner_rank"] for r in results])

    print(f"\n{'='*56}")
    print(f"LOOCV Summary")
    print(f"{'='*56}")
    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"  {mark} {r['test_year']} | "
              f"predicted: {r['predicted']:40s} | "
              f"actual: {r['winner']}")

    print(f"\nOverall Top-1  : {top1}/{n} = {top1/n:.1%}  (random: ~12.5%)")
    print(f"Overall Top-3  : {top3}/{n} = {top3/n:.1%}  (random: ~33.3%)")
    print(f"MRR            : {mrr:.4f}  (random: ~0.3083)")

    return results


# ── RUN ───────────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Need to move these to earlier preprocessing
imdb_df["rating"] = pd.to_numeric(imdb_df["rating"], errors="coerce")

# Les Mis fix
imdb_df["film_title"]       = imdb_df["film_title"].str.encode("utf-8", errors="ignore").str.decode("utf-8")
metacritic_df["film_title"] = metacritic_df["film_title"].str.encode("utf-8", errors="ignore").str.decode("utf-8")

# Re-apply title mapping
imdb_df["film_title"]       = imdb_df["film_title"].replace(title_mapping)
metacritic_df["film_title"] = metacritic_df["film_title"].replace(title_mapping)

# Filter to confirmed year window
nominees_df_filtered = nominees_df[
    (nominees_df["ceremony_year"] >= 2012) &
    (nominees_df["ceremony_year"] <= 2020)
].copy()

smoke_test(nominees_df_filtered, metacritic_df, imdb_df, device=device)

# HYPERPARAM TUNE HERE
results = run_loocv(
        nominees_df        = nominees_df_filtered,
        metacritic_df      = metacritic_df,
        imdb_df            = imdb_df,
        num_epochs         = 10,
        lr                 = lr,
        regularizer        = regularizer,
        max_critic_reviews = 40,
        max_imdb_reviews   = 50,
        max_len            = 128,
        patience           = 5,
        device             = device
    )

**Model 1 (RoBERTA base) Intermediary Fix:**

1.   **Original RoBERTa:** encoder lives inside model, runs every epoch. Slow because RoBERTa re-encodes all reviews on every forward pass despite frozen weights never changing.
2.   **Revised RoBERTa:** since frozen encoder produces identical output each epoch, run it once and save (once instead of 100 times per fold). This optimizes speed so epochs only run on attention layers, not RoBERTa.
3. **Real problem:** ran similarity analysis and found RoBERTa embeddings are 0.997 similar regardless of caching. Caching made it fast but the embeddings themselves were useless.
4. **Sentence transformers application** — replaced precompute_and_cache() with precompute_st_embeddings(). Same caching concept, just different encoder where similarities are now 0.60-0.80. OscarEmbeddingDataset (wraps cache as iterable for DataLoader) is unchanged because cache structure is identical.


In [ ]:
# ── STEP 1: ENCODER ONLY — used once to precompute ────────────────────────────
# Extract just the RoBERTa encoding portion from OscarPredictorAttention
# so we can run it once and cache results

class RoBERTaEncoder(nn.Module):
    """Lightweight wrapper — just the frozen encoder + mean pool."""
    def __init__(self, model_name="roberta-base"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        for param in self.encoder.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def encode_reviews(self, input_ids, attention_mask):
        """
        input_ids      : [num_reviews, seq_len]
        attention_mask : [num_reviews, seq_len]
        returns        : [num_reviews, 768]
        """
        outputs  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden   = outputs.last_hidden_state
        mask_exp = attention_mask.unsqueeze(-1).float()
        pooled   = (hidden * mask_exp).sum(dim=1) / \
                   mask_exp.sum(dim=1).clamp(min=1e-8)
        return pooled  # [num_reviews, 768]

# ── STEP 2: PRECOMPUTE AND CACHE ───────────────────────────────────────────────
def precompute_and_cache(nominees_df, metacritic_df, imdb_df,
                          tokenizer, device,
                          max_critic_reviews=40, max_imdb_reviews=50,
                          max_len=128, save_path="oscar_embeddings.pt"):
    """
    Runs RoBERTa once over all reviews for all films across all years.
    Saves a cache of [num_reviews, 768] tensors per film per year.
    Subsequent training epochs load from cache — no RoBERTa forward passes.
    """
    print("Precomputing RoBERTa embeddings...")
    encoder = RoBERTaEncoder().to(device)
    encoder.eval()

    # Reuse existing dataset class just for tokenization
    full_dataset = OscarCohortDataset(
        nominees_df, metacritic_df, imdb_df, tokenizer,
        max_critic_reviews=max_critic_reviews,
        max_imdb_reviews=max_imdb_reviews,
        max_len=max_len
    )

    cache = {}

    for cohort_idx in range(len(full_dataset)):
        sample = full_dataset[cohort_idx]
        year   = sample["year"].item() if torch.is_tensor(sample["year"]) \
                 else int(sample["year"])
        films  = sample["films"]

        print(f"  Encoding year {year} — {len(films)} films")
        cache[year] = []

        # critic_input_ids: [n_films, max_critic_reviews, seq_len]
        critic_ids  = sample["critic_input_ids"]
        critic_mask = sample["critic_attention_mask"]
        imdb_ids    = sample["imdb_input_ids"]
        imdb_mask   = sample["imdb_attention_mask"]

        for i, film in enumerate(films):
            # Encode critic reviews for this film
            c_ids  = critic_ids[i].to(device)   # [max_critic_reviews, seq_len]
            c_mask = critic_mask[i].to(device)
            c_emb  = encoder.encode_reviews(c_ids, c_mask)  # [max_critic, 768]

            # Encode IMDb reviews for this film
            i_ids  = imdb_ids[i].to(device)     # [max_imdb_reviews, seq_len]
            i_mask = imdb_mask[i].to(device)
            i_emb  = encoder.encode_reviews(i_ids, i_mask)  # [max_imdb, 768]

            # Also cache the padding masks for attention layers
            # True = padded review (all zeros), used by TransformerEncoder
            c_pad_mask = (c_mask.sum(dim=-1) == 0)  # [max_critic_reviews]
            i_pad_mask = (i_mask.sum(dim=-1) == 0)  # [max_imdb_reviews]

            cache[year].append({
                "film":          film,
                "critic_emb":    c_emb.cpu(),        # [max_critic, 768]
                "imdb_emb":      i_emb.cpu(),        # [max_imdb, 768]
                "critic_pad":    c_pad_mask.cpu(),   # [max_critic]
                "imdb_pad":      i_pad_mask.cpu(),   # [max_imdb]
                "winner_idx":    sample["winner_idx"].item(),
            })

    torch.save(cache, save_path)
    # print(f"Saved to Drive: {save_path_drive}")
    # print(f"Saved embeddings → {save_path}")
    print(f"Years cached: {sorted(cache.keys())}")
    return cache

# ── STEP 3: CACHED DATASET (independent of RoBERTA vs sentence transformer) ────────────────────────────────────────────────────
class OscarEmbeddingDataset(Dataset):
    """
    Loads precomputed embeddings instead of raw text.
    Each item is one cohort year — [n_films, max_reviews, 768] tensors.
    Training epochs are now seconds not minutes.
    """
    def __init__(self, cache, years):
        self.cohorts = []

        for year in years:
            if year not in cache:
                print(f"Warning: year {year} not in cache, skipping")
                continue

            film_data  = cache[year]
            n_films    = len(film_data)
            winner_idx = film_data[0]["winner_idx"]  # same for all films in year

            critic_embs = torch.stack([f["critic_emb"] for f in film_data])  # [n_films, max_critic, 768]
            imdb_embs   = torch.stack([f["imdb_emb"]   for f in film_data])  # [n_films, max_imdb,   768]
            critic_pads = torch.stack([f["critic_pad"]  for f in film_data])  # [n_films, max_critic]
            imdb_pads   = torch.stack([f["imdb_pad"]    for f in film_data])  # [n_films, max_imdb]
            films       = [f["film"] for f in film_data]

            self.cohorts.append({
                "year":        year,
                "films":       films,
                "critic_embs": critic_embs,
                "imdb_embs":   imdb_embs,
                "critic_pads": critic_pads,
                "imdb_pads":   imdb_pads,
                "winner_idx":  torch.tensor(winner_idx, dtype=torch.long),
            })

    def __len__(self):
        return len(self.cohorts)

    def __getitem__(self, idx):
        return self.cohorts[idx]

# ── STEP 4: MODIFIED MODEL — takes embeddings not raw tokens ──────────────────
class OscarPredictorAttentionCached(nn.Module):
    """
    Same architecture as OscarPredictorAttention but takes precomputed
    [n_films, num_reviews, 768] embeddings instead of raw token ids.
    RoBERTa encoding removed — handled by precompute step.
    """
    def __init__(self, hidden_dim=768, num_heads=8,
                 num_self_attn_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Self-attention over reviews within each stream
        critic_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.critic_self_attn = nn.TransformerEncoder(
            critic_layer, num_layers=num_self_attn_layers
        )

        imdb_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.imdb_self_attn = nn.TransformerEncoder(
            imdb_layer, num_layers=num_self_attn_layers
        )

        # Cross-attention between streams
        self.cross_attn_critic_to_imdb = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.cross_attn_imdb_to_critic = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )

        self.norm_critic_self  = nn.LayerNorm(hidden_dim)
        self.norm_imdb_self    = nn.LayerNorm(hidden_dim)
        self.norm_critic_cross = nn.LayerNorm(hidden_dim)
        self.norm_imdb_cross   = nn.LayerNorm(hidden_dim)

        # Fusion
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 4, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, critic_embs, critic_pad_mask,
                      imdb_embs,   imdb_pad_mask):
        """
        critic_embs    : [n_films, max_critic_reviews, 768]
        critic_pad_mask: [n_films, max_critic_reviews]  True=padded
        imdb_embs      : [n_films, max_imdb_reviews,   768]
        imdb_pad_mask  : [n_films, max_imdb_reviews]    True=padded
        returns        : [n_films, 1] raw logits
        """
        h_critic = critic_embs
        h_imdb   = imdb_embs

        # Self-attention within each stream
        h_critic_self = self.critic_self_attn(
            h_critic, src_key_padding_mask=critic_pad_mask
        )
        h_critic = self.norm_critic_self(h_critic + h_critic_self)

        h_imdb_self = self.imdb_self_attn(
            h_imdb, src_key_padding_mask=imdb_pad_mask
        )
        h_imdb = self.norm_imdb_self(h_imdb + h_imdb_self)

        # Cross-attention between streams
        h_critic_cross, _ = self.cross_attn_critic_to_imdb(
            query=h_critic, key=h_imdb, value=h_imdb,
            key_padding_mask=imdb_pad_mask
        )
        h_critic = self.norm_critic_cross(h_critic + h_critic_cross)

        h_imdb_cross, _ = self.cross_attn_imdb_to_critic(
            query=h_imdb, key=h_critic, value=h_critic,
            key_padding_mask=critic_pad_mask
        )
        h_imdb = self.norm_imdb_cross(h_imdb + h_imdb_cross)

        # Pool to one vector per film
        c_critic = h_critic.mean(dim=1)   # [n_films, 768]
        c_imdb   = h_imdb.mean(dim=1)     # [n_films, 768]

        diff    = c_critic - c_imdb
        product = c_critic * c_imdb
        fused   = torch.cat([c_critic, c_imdb, diff, product], dim=-1)

        return self.scorer(fused)         # [n_films, 1]

Precomputing RoBERTa embeddings...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dataset built: 9 cohorts
  Encoding year 2012 — 9 films
  Encoding year 2013 — 9 films
  Encoding year 2014 — 9 films
  Encoding year 2015 — 8 films
  Encoding year 2016 — 8 films
  Encoding year 2017 — 9 films
  Encoding year 2018 — 9 films
  Encoding year 2019 — 8 films
  Encoding year 2020 — 9 films
Years cached: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

Fold: test year = 2012
  Epoch 010 | loss 2.2670 | train acc 1/8
  Epoch 020 | loss 2.3212 | train acc 1/8
  Epoch 030 | loss 2.2237 | train acc 0/8
  Epoch 040 | loss 2.2275 | train acc 1/8
  Epoch 050 | loss 2.1296 | train acc 0/8

  Results for 2012:
    The Artist (I)                                0.111 ← winner ← predicted
    The Descendants                               0.111
    Extremely Loud & Incredibly Close             0.111
    The Help                                      0.111
    Hugo                                          0.111
    Midnight in Paris                             0.111
    Moneyball  

**Continued Model 1 code (split out for readability)**

This is mostly revised training functions

In [ ]:
# ── STEP 5: FAST TRAINING LOOP ────────────────────────────────────────────────
def train_step_cached(model, optimizer, batch, device):
    model.train()
    optimizer.zero_grad()

    critic_embs = batch["critic_embs"].squeeze(0).to(device)   # [n_films, max_critic, 768]
    imdb_embs   = batch["imdb_embs"].squeeze(0).to(device)
    critic_pads = batch["critic_pads"].squeeze(0).to(device)   # [n_films, max_critic]
    imdb_pads   = batch["imdb_pads"].squeeze(0).to(device)
    winner_idx  = batch["winner_idx"].squeeze().to(device)

    logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)

    loss = nn.CrossEntropyLoss()(
        logits.squeeze(-1).unsqueeze(0),
        winner_idx.unsqueeze(0)
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    probs     = torch.softmax(logits.squeeze(-1), dim=0)
    predicted = probs.argmax().item()
    correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()

def eval_step_cached(model, batch, device):
    model.eval()
    with torch.no_grad():
        critic_embs = batch["critic_embs"].squeeze(0).to(device)
        imdb_embs   = batch["imdb_embs"].squeeze(0).to(device)
        critic_pads = batch["critic_pads"].squeeze(0).to(device)
        imdb_pads   = batch["imdb_pads"].squeeze(0).to(device)
        winner_idx  = batch["winner_idx"].squeeze().to(device)

        logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)

        loss = nn.CrossEntropyLoss()(
            logits.squeeze(-1).unsqueeze(0),
            winner_idx.unsqueeze(0)
        )

        probs     = torch.softmax(logits.squeeze(-1), dim=0)
        predicted = probs.argmax().item()
        correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()

# ── STEP 6: FAST LOOCV ────────────────────────────────────────────────────────
def run_loocv_cached(cache, nominees_df,
                     num_epochs=50, lr=3e-4,
                     weight_decay=0.01,
                     patience=8,
                     device="cpu"):

    all_years = sorted(cache.keys())
    results   = []

    for test_year in all_years:
        print(f"\n{'='*56}")
        print(f"Fold: test year = {test_year}")
        print(f"{'='*56}")

        train_years = [y for y in all_years if y != test_year]

        train_dataset = OscarEmbeddingDataset(cache, train_years)
        test_dataset  = OscarEmbeddingDataset(cache, [test_year])

        train_loader  = DataLoader(train_dataset, batch_size=1, shuffle=True)
        test_loader   = DataLoader(test_dataset,  batch_size=1, shuffle=False)

        model     = OscarPredictorAttentionCached().to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs
        )
        # Early stopping state
        best_loss  = float("inf")
        no_improve = 0
        best_state = None

        for epoch in range(num_epochs):
            ...
            if mean_loss < best_loss - 1e-4:
                best_loss  = mean_loss
                no_improve = 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break

        # Restore best weights before eval
        if best_state is not None:
            model.load_state_dict(best_state)

        # Evaluate
        print(f"\n  Results for {test_year}:")
        for batch in test_loader:
            loss, correct, probs = eval_step_cached(model, batch, device)
            films      = batch["films"]
            winner_idx = batch["winner_idx"].squeeze().item()
            pred_idx   = probs.index(max(probs))

            for i, (film, prob) in enumerate(zip(films, probs)):
                w = " ← winner"    if i == winner_idx else ""
                p = " ← predicted" if i == pred_idx   else ""
                print(f"    {film[0]:45s} {prob:.3f}{w}{p}")

            # Compute winner rank for MRR
            ranked_films = [f for f, p in sorted(
                zip([fi[0] for fi in films], probs),
                key=lambda x: x[1], reverse=True
            )]
            winner_rank = ranked_films.index(films[winner_idx][0]) + 1

            results.append({
                "test_year":   test_year,
                "correct":     correct,
                "winner":      films[winner_idx][0],
                "predicted":   films[pred_idx][0],
                "probs":       probs,
                "films":       [f[0] for f in films],
                "winner_rank": winner_rank,
            })

    # Summary
    print(f"\n{'='*56}")
    print(f"LOOCV Summary (cached embeddings)")
    print(f"{'='*56}")
    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"  {mark} {r['test_year']} | "
              f"predicted: {r['predicted']:40s} | "
              f"actual: {r['winner']}")

    n       = len(results)
    top1    = sum(r["correct"] for r in results)
    top3    = sum(1 for r in results
                  if r["winner"] in [f for f, p in sorted(
                      zip(r["films"], r["probs"]),
                      key=lambda x: x[1], reverse=True)][:3])
    mrr     = np.mean([1.0 / r["winner_rank"] for r in results])

    accuracy = top1 / n
    print(f"\nOverall Top-1  : {top1}/{n} = {accuracy:.1%}  (random: ~12.5%)")
    print(f"Overall Top-3  : {top3}/{n} = {top3/n:.1%}  (random: ~33.3%)")
    print(f"MRR            : {mrr:.4f}  (random: ~0.3083)")

    return results

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Precompute once — takes a few minutes, saves to disk
cache = precompute_and_cache(
    nominees_df    = nominees_df_filtered,
    metacritic_df  = metacritic_df,
    imdb_df        = imdb_df,
    tokenizer      = tokenizer,
    device         = device,
    max_critic_reviews = 40,
    max_imdb_reviews   = 50,
    max_len            = 128,
    save_path      = "oscar_embeddings.pt"
)

# If cache already exists on disk, just load it
# cache = torch.load("oscar_embeddings.pt")

**Cached Model 1 (RoBERTa) Training**

In [ ]:
# Train with 50 epochs — each epoch is now seconds not minutes
results_cached = run_loocv_cached(
    cache       = cache,
    nominees_df = nominees_df_filtered,
    num_epochs  = 50,
    lr          = 3e-4,
    device      = device
)

**Discovery of Embeddings Issue**

In [ ]:
# Figure out problem is the embeddings themselves or just the training regime

cohort_2012 = cache[2012]

print("Mean pooled critic embedding similarities (2012 cohort):")
vecs = [c["critic_emb"].mean(dim=0) for c in cohort_2012]

for i in range(len(vecs)):
    for j in range(i+1, len(vecs)):
        sim = torch.nn.functional.cosine_similarity(
            vecs[i].unsqueeze(0), vecs[j].unsqueeze(0)
        ).item()
        print(f"  {cohort_2012[i]['film']:35s} vs {cohort_2012[j]['film']:35s} : {sim:.4f}")

Mean pooled critic embedding similarities (2012 cohort):
  The Artist (I)                      vs The Descendants                     : 0.9989
  The Artist (I)                      vs Extremely Loud & Incredibly Close   : 0.9987
  The Artist (I)                      vs The Help                            : 0.9986
  The Artist (I)                      vs Hugo                                : 0.9990
  The Artist (I)                      vs Midnight in Paris                   : 0.9988
  The Artist (I)                      vs Moneyball                           : 0.9988
  The Artist (I)                      vs The Tree of Life                    : 0.9984
  The Artist (I)                      vs War Horse                           : 0.9992
  The Descendants                     vs Extremely Loud & Incredibly Close   : 0.9988
  The Descendants                     vs The Help                            : 0.9991
  The Descendants                     vs Hugo                                : 0.99

In [ ]:
class OscarPredictorAttention(nn.Module):
    """
    Dual-stream model with real multi-head self-attention and cross-attention.
    Top 3 RoBERTa layers unfrozen — allows encoder to learn film-discriminative
    representations rather than producing near-identical embeddings for all films.

    Pipeline per film:
      1. RoBERTa (partially unfrozen) encodes each review
      2. Mean pool tokens → one vector per review
      3. Self-attention across reviews within each stream
      4. Cross-attention between streams (critic ↔ IMDb)
      5. Mean pool reviews → one vector per stream
      6. Fusion → single film logit
    """
    def __init__(
        self,
        model_name="roberta-base",
        hidden_dim=768,
        num_heads=8,
        num_self_attn_layers=2,
        dropout=0.3,
        unfreeze_last_n_layers=3
    ):
        super(OscarPredictorAttention, self).__init__()

        self.hidden_dim = hidden_dim

        # Shared RoBERTa encoder
        self.encoder = AutoModel.from_pretrained(model_name)

        # Freeze all layers first
        for param in self.encoder.parameters():
            param.requires_grad = False

        # Unfreeze top N layers — enough to learn discriminative representations
        # without destroying pretrained knowledge in lower layers
        for layer in self.encoder.encoder.layer[-unfreeze_last_n_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Unfreeze pooler
        for param in self.encoder.pooler.parameters():
            param.requires_grad = True

        # Print trainable parameter count as sanity check
        total     = sum(p.numel() for p in self.encoder.parameters())
        trainable = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
        print(f"RoBERTa: {trainable:,} / {total:,} params unfrozen "
              f"({100*trainable/total:.1f}%)")

        # Self-attention over reviews within each stream
        critic_self_attn_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.critic_self_attn = nn.TransformerEncoder(
            critic_self_attn_layer,
            num_layers=num_self_attn_layers
        )

        imdb_self_attn_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.imdb_self_attn = nn.TransformerEncoder(
            imdb_self_attn_layer,
            num_layers=num_self_attn_layers
        )

        # Cross-attention between streams
        self.cross_attn_critic_to_imdb = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.cross_attn_imdb_to_critic = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm_critic_self  = nn.LayerNorm(hidden_dim)
        self.norm_imdb_self    = nn.LayerNorm(hidden_dim)
        self.norm_critic_cross = nn.LayerNorm(hidden_dim)
        self.norm_imdb_cross   = nn.LayerNorm(hidden_dim)

        # Fusion
        fusion_input_dim = hidden_dim * 4

        self.scorer = nn.Sequential(
            nn.Linear(fusion_input_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def encode_reviews(self, input_ids, attention_mask):
        """
        input_ids      : [batch, num_reviews, seq_len]
        attention_mask : [batch, num_reviews, seq_len]
        returns        : [batch, num_reviews, hidden_dim]
        """
        batch_size, num_reviews, seq_len = input_ids.shape

        ids_flat  = input_ids.reshape(-1, seq_len)
        mask_flat = attention_mask.reshape(-1, seq_len)

        # No torch.no_grad() — top layers are now trainable
        outputs = self.encoder(input_ids=ids_flat, attention_mask=mask_flat)

        hidden   = outputs.last_hidden_state
        mask_exp = mask_flat.unsqueeze(-1).float()
        pooled   = (hidden * mask_exp).sum(dim=1) / \
                   mask_exp.sum(dim=1).clamp(min=1e-8)

        return pooled.reshape(batch_size, num_reviews, -1)

    def mean_pool_reviews(self, x):
        return x.mean(dim=1)

    def forward(
        self,
        critic_input_ids,    critic_attention_mask,
        imdb_input_ids,      imdb_attention_mask,
    ):
        # 1. Encode
        h_critic = self.encode_reviews(critic_input_ids, critic_attention_mask)
        h_imdb   = self.encode_reviews(imdb_input_ids,   imdb_attention_mask)

        # 2. Self-attention
        critic_pad_mask = (critic_attention_mask.sum(dim=-1) == 0)
        imdb_pad_mask   = (imdb_attention_mask.sum(dim=-1)   == 0)

        h_critic_self = self.critic_self_attn(
            h_critic, src_key_padding_mask=critic_pad_mask
        )
        h_critic = self.norm_critic_self(h_critic + h_critic_self)

        h_imdb_self = self.imdb_self_attn(
            h_imdb, src_key_padding_mask=imdb_pad_mask
        )
        h_imdb = self.norm_imdb_self(h_imdb + h_imdb_self)

        # 3. Cross-attention
        h_critic_cross, _ = self.cross_attn_critic_to_imdb(
            query=h_critic, key=h_imdb, value=h_imdb,
            key_padding_mask=imdb_pad_mask
        )
        h_critic = self.norm_critic_cross(h_critic + h_critic_cross)

        h_imdb_cross, _ = self.cross_attn_imdb_to_critic(
            query=h_imdb, key=h_critic, value=h_critic,
            key_padding_mask=critic_pad_mask
        )
        h_imdb = self.norm_imdb_cross(h_imdb + h_imdb_cross)

        # 4. Pool
        c_critic = self.mean_pool_reviews(h_critic)
        c_imdb   = self.mean_pool_reviews(h_imdb)

        # 5. Fusion
        diff    = c_critic - c_imdb
        product = c_critic * c_imdb
        fused   = torch.cat([c_critic, c_imdb, diff, product], dim=-1)

        return self.scorer(fused)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer


def check_embedding_similarity(model, batch, device):
    """
    Run after first epoch to verify unfrozen layers are producing
    discriminative embeddings. Should see similarities drop from 0.999
    down to 0.85 or lower if unfreezing is working.
    """
    model.eval()
    with torch.no_grad():
        critic_ids  = batch["critic_input_ids"].squeeze(0).to(device)
        critic_mask = batch["critic_attention_mask"].squeeze(0).to(device)
        imdb_ids    = batch["imdb_input_ids"].squeeze(0).to(device)
        imdb_mask   = batch["imdb_attention_mask"].squeeze(0).to(device)

        h_critic = model.encode_reviews(critic_ids, critic_mask)  # [n_films, num_reviews, 768]
        h_imdb   = model.encode_reviews(imdb_ids,   imdb_mask)

        # Mean pool reviews → one vector per film
        c_critic = h_critic.mean(dim=1)  # [n_films, 768]
        c_imdb   = h_imdb.mean(dim=1)

    print("\nCritic embedding similarities after this epoch:")
    films = batch["films"]
    for i in range(len(films)):
        for j in range(i+1, len(films)):
            sim = torch.nn.functional.cosine_similarity(
                c_critic[i].unsqueeze(0),
                c_critic[j].unsqueeze(0)
            ).item()
            print(f"  {films[i][0]:30s} vs {films[j][0]:30s} : {sim:.4f}")

    print("\nIMDb embedding similarities after this epoch:")
    for i in range(len(films)):
        for j in range(i+1, len(films)):
            sim = torch.nn.functional.cosine_similarity(
                c_imdb[i].unsqueeze(0),
                c_imdb[j].unsqueeze(0)
            ).item()
            print(f"  {films[i][0]:30s} vs {films[j][0]:30s} : {sim:.4f}")

    model.train()

# ── DATASET ───────────────────────────────────────────────────────────────────
class OscarCohortDataset(Dataset):
    def __init__(self, nominees_df, metacritic_df, imdb_df,
                 tokenizer, max_critic_reviews=40, max_imdb_reviews=50,
                 max_len=128, seed=42):

        self.tokenizer          = tokenizer
        self.max_critic_reviews = max_critic_reviews
        self.max_imdb_reviews   = max_imdb_reviews
        self.max_len            = max_len
        self.seed               = seed
        self.cohorts            = []

        for year, year_nominees in nominees_df.groupby("ceremony_year"):
            films      = year_nominees["film_title"].tolist()
            winner_row = year_nominees[year_nominees["winner"] == 1]

            if winner_row.empty:
                print(f"  Warning: no winner found for {year}, skipping")
                continue

            winner_title = winner_row["film_title"].iloc[0]

            if winner_title not in films:
                print(f"  Warning: winner {winner_title} not in films for {year}, skipping")
                continue

            winner_idx = films.index(winner_title)

            critic_reviews_per_film = []
            imdb_reviews_per_film   = []

            for film in films:
                # Metacritic — clean_text column
                critic_pool = metacritic_df[
                    (metacritic_df["film_title"]    == film) &
                    (metacritic_df["ceremony_year"] == year)
                ]["clean_text"].dropna().tolist()

                # IMDb — clean_text column, stratified sample
                imdb_pool_df = imdb_df[
                    (imdb_df["film_title"]    == film) &
                    (imdb_df["ceremony_year"] == year)
                ].dropna(subset=["clean_text"])

                if len(imdb_pool_df) > self.max_imdb_reviews and "rating" in imdb_pool_df.columns:
                    imdb_pool = self._stratified_sample(
                        imdb_pool_df, n=self.max_imdb_reviews
                    )["clean_text"].tolist()
                else:
                    imdb_pool = imdb_pool_df["clean_text"].tolist()[:self.max_imdb_reviews]

                # Fallbacks for missing reviews
                if not critic_pool:
                    print(f"  Warning: no critic reviews for {film} ({year})")
                    critic_pool = ["no critic reviews available"]

                if not imdb_pool:
                    print(f"  Warning: no IMDb reviews for {film} ({year})")
                    imdb_pool = ["no user reviews available"]

                critic_reviews_per_film.append(critic_pool)
                imdb_reviews_per_film.append(imdb_pool)

            self.cohorts.append({
                "year":           year,
                "films":          films,
                "critic_reviews": critic_reviews_per_film,
                "imdb_reviews":   imdb_reviews_per_film,
                "winner_idx":     winner_idx,
            })

        print(f"Dataset built: {len(self.cohorts)} cohorts")

    def _stratified_sample(self, df, n):
        df = df.copy()
        df["bucket"] = pd.cut(
            df["rating"],
            bins=[0, 4, 7, 10],
            labels=["negative", "mixed", "positive"],
            include_lowest=True
        )
        frames = []
        for bucket, prop in df["bucket"].value_counts(normalize=True).items():
            k          = max(1, round(prop * n))
            bucket_df  = df[df["bucket"] == bucket]
            k          = min(k, len(bucket_df))
            frames.append(bucket_df.sample(n=k, random_state=self.seed))
        return pd.concat(frames).head(n)

    def _tokenize(self, reviews, max_reviews):
        reviews = [r for r in reviews if isinstance(r, str) and len(r.strip()) > 0]
        reviews = reviews[:max_reviews]
        if not reviews:
            reviews = [""]

        encoded = self.tokenizer(
            reviews,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        ids  = encoded["input_ids"]
        mask = encoded["attention_mask"]

        # Pad up to max_reviews if fewer reviews exist
        n = ids.shape[0]
        if n < max_reviews:
            pad_ids  = torch.zeros(max_reviews - n, self.max_len, dtype=torch.long)
            pad_mask = torch.zeros(max_reviews - n, self.max_len, dtype=torch.long)
            ids  = torch.cat([ids,  pad_ids],  dim=0)
            mask = torch.cat([mask, pad_mask], dim=0)

        return ids, mask

    def __len__(self):
        return len(self.cohorts)

    def __getitem__(self, idx):
        cohort  = self.cohorts[idx]
        n_films = len(cohort["films"])

        critic_ids_list, critic_mask_list = [], []
        imdb_ids_list,   imdb_mask_list   = [], []

        for i in range(n_films):
            c_ids, c_mask = self._tokenize(cohort["critic_reviews"][i], self.max_critic_reviews)
            i_ids, i_mask = self._tokenize(cohort["imdb_reviews"][i],   self.max_imdb_reviews)

            critic_ids_list.append(c_ids)
            critic_mask_list.append(c_mask)
            imdb_ids_list.append(i_ids)
            imdb_mask_list.append(i_mask)

        return {
            "year":                  cohort["year"],
            "films":                 cohort["films"],
            "critic_input_ids":      torch.stack(critic_ids_list),    # [n_films, 40, 128]
            "critic_attention_mask": torch.stack(critic_mask_list),
            "imdb_input_ids":        torch.stack(imdb_ids_list),      # [n_films, 50, 128]
            "imdb_attention_mask":   torch.stack(imdb_mask_list),
            "winner_idx":            torch.tensor(cohort["winner_idx"], dtype=torch.long),
        }


# ── TRAIN / EVAL STEPS ────────────────────────────────────────────────────────
def train_step(model, optimizer, batch, device):
    model.train()
    optimizer.zero_grad()

    critic_ids  = batch["critic_input_ids"].squeeze(0).to(device)
    critic_mask = batch["critic_attention_mask"].squeeze(0).to(device)
    imdb_ids    = batch["imdb_input_ids"].squeeze(0).to(device)
    imdb_mask   = batch["imdb_attention_mask"].squeeze(0).to(device)
    winner_idx  = batch["winner_idx"].squeeze().to(device)  # flatten to scalar

    logits = model(critic_ids, critic_mask, imdb_ids, imdb_mask)

    loss = nn.CrossEntropyLoss()(
        logits.squeeze(-1).unsqueeze(0),   # [1, n_films]
        winner_idx.unsqueeze(0)            # [1]
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    probs     = torch.softmax(logits.squeeze(-1), dim=0)
    predicted = probs.argmax().item()
    correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


def eval_step(model, batch, device):
    model.eval()
    with torch.no_grad():
        critic_ids  = batch["critic_input_ids"].squeeze(0).to(device)
        critic_mask = batch["critic_attention_mask"].squeeze(0).to(device)
        imdb_ids    = batch["imdb_input_ids"].squeeze(0).to(device)
        imdb_mask   = batch["imdb_attention_mask"].squeeze(0).to(device)
        winner_idx  = batch["winner_idx"].squeeze().to(device)  # flatten to scalar

        logits = model(critic_ids, critic_mask, imdb_ids, imdb_mask)

        loss = nn.CrossEntropyLoss()(
            logits.squeeze(-1).unsqueeze(0),
            winner_idx.unsqueeze(0)
        )

        probs     = torch.softmax(logits.squeeze(-1), dim=0)
        predicted = probs.argmax().item()
        correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


# ── SANITY CHECK ──────────────────────────────────────────────────────────────
def smoke_test(nominees_df, metacritic_df, imdb_df, device="cpu"):
    """
    Run one forward pass on one cohort before full training.
    Catches shape errors, missing columns, and device issues early.
    """
    print("Running smoke test...")
    tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    dataset   = OscarCohortDataset(
        nominees_df, metacritic_df, imdb_df, tokenizer,
        max_critic_reviews=5, max_imdb_reviews=5, max_len=64  # tiny for speed
    )

    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    batch  = next(iter(loader))

    print(f"  Year: {batch['year']}")
    print(f"  Films: {batch['films']}")
    print(f"  Winner idx: {batch['winner_idx'].item()}")
    print(f"  critic_input_ids shape: {batch['critic_input_ids'].shape}")
    print(f"  imdb_input_ids shape:   {batch['imdb_input_ids'].shape}")

    model  = OscarPredictorAttention().to(device)
    logits = model(
        batch["critic_input_ids"].squeeze(0).to(device),
        batch["critic_attention_mask"].squeeze(0).to(device),
        batch["imdb_input_ids"].squeeze(0).to(device),
        batch["imdb_attention_mask"].squeeze(0).to(device),
    )

    print(f"  Output logits shape: {logits.shape}")   # should be [n_films, 1]
    print(f"  Logits: {logits.squeeze(-1).tolist()}")
    print("Smoke test passed.")
    return True


# ── LOOCV TRAINING ────────────────────────────────────────────────────────────
def run_loocv(nominees_df, metacritic_df, imdb_df,
              num_epochs=10, lr=1e-4,
              max_critic_reviews=40, max_imdb_reviews=50,
              max_len=128, device="cpu"):

    tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    all_years = sorted(nominees_df["ceremony_year"].unique())
    results   = []

    for test_year in all_years:
        print(f"\n{'='*56}")
        print(f"Fold: test year = {test_year}")
        print(f"{'='*56}")

        # Split by year
        train_nom  = nominees_df[nominees_df["ceremony_year"] != test_year]
        test_nom   = nominees_df[nominees_df["ceremony_year"] == test_year]
        train_meta = metacritic_df[metacritic_df["ceremony_year"] != test_year]
        test_meta  = metacritic_df[metacritic_df["ceremony_year"] == test_year]
        train_imdb = imdb_df[imdb_df["ceremony_year"] != test_year]
        test_imdb  = imdb_df[imdb_df["ceremony_year"] == test_year]

        # Build datasets
        train_dataset = OscarCohortDataset(
            train_nom, train_meta, train_imdb, tokenizer,
            max_critic_reviews=max_critic_reviews,
            max_imdb_reviews=max_imdb_reviews,
            max_len=max_len
        )
        test_dataset = OscarCohortDataset(
            test_nom, test_meta, test_imdb, tokenizer,
            max_critic_reviews=max_critic_reviews,
            max_imdb_reviews=max_imdb_reviews,
            max_len=max_len
        )

        train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
        test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)

        # Fresh model per fold
        model     = OscarPredictorAttention().to(device)
        # In run_loocv — two different learning rates
        # BERT layers need much lower LR than the fresh attention/fusion layers
        optimizer = torch.optim.AdamW([
            {"params": model.encoder.parameters(),       "lr": 2e-5},  # unfrozen BERT layers
            {"params": model.critic_self_attn.parameters(), "lr": 1e-4},
            {"params": model.imdb_self_attn.parameters(),   "lr": 1e-4},
            {"params": model.cross_attn_critic_to_imdb.parameters(), "lr": 1e-4},
            {"params": model.cross_attn_imdb_to_critic.parameters(), "lr": 1e-4},
            {"params": model.scorer.parameters(),           "lr": 1e-4},
        ], weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs
        )

        # Training epochs
        for epoch in range(num_epochs):
          losses, corrects = [], []
          for batch in train_loader:
              loss, correct, _ = train_step(model, optimizer, batch, device)
              losses.append(loss)
              corrects.append(correct)

          scheduler.step()
          print(f"  Epoch {epoch+1:02d} | loss {np.mean(losses):.4f} | "
                f"train acc {sum(corrects)}/{len(corrects)}")

          # Check similarities after epoch 1 and 5
          if epoch in [0, 4]:
              sample_batch = next(iter(train_loader))
              check_embedding_similarity(model, sample_batch, device)

        # Evaluate on held-out year
        print(f"\n  Results for {test_year}:")
        for batch in test_loader:
            loss, correct, probs = eval_step(model, batch, device)
            films      = batch["films"]
            winner_idx = batch["winner_idx"].item()
            pred_idx   = probs.index(max(probs))

            for i, (film, prob) in enumerate(zip(films, probs)):
                winner_marker = " ← winner"    if i == winner_idx else ""
                pred_marker   = " ← predicted" if i == pred_idx   else ""
                print(f"    {film[0]:45s} {prob:.3f}{winner_marker}{pred_marker}")

            results.append({
                "test_year": test_year,
                "correct":   correct,
                "winner":    films[winner_idx][0],
                "predicted": films[pred_idx][0],
            })

    # Final summary
    print(f"\n{'='*56}")
    print(f"LOOCV Summary")
    print(f"{'='*56}")
    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"  {mark} {r['test_year']} | "
              f"predicted: {r['predicted']:40s} | "
              f"actual: {r['winner']}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\nOverall: {sum(r['correct'] for r in results)}/{len(results)} = {accuracy:.1%}")
    print(f"Random baseline: ~12.5%")

    return results


# ── RUN ───────────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Always run smoke test first
smoke_test(nominees_df_filtered, metacritic_df, imdb_df, device=device)

# Then full training
results = run_loocv(
    nominees_df        = nominees_df_filtered,
    metacritic_df      = metacritic_df,
    imdb_df            = imdb_df,
    num_epochs         = 30,
    max_critic_reviews = 15,
    max_imdb_reviews   = 20,
    max_len            = 128,
    device             = device
)

Device: cuda
Running smoke test...
Dataset built: 9 cohorts
  Year: tensor([2012])
  Films: [('The Artist (I)',), ('The Descendants',), ('Extremely Loud & Incredibly Close',), ('The Help',), ('Hugo',), ('Midnight in Paris',), ('Moneyball',), ('The Tree of Life',), ('War Horse',)]
  Winner idx: 0
  critic_input_ids shape: torch.Size([1, 9, 5, 64])
  imdb_input_ids shape:   torch.Size([1, 9, 5, 64])


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa: 21,854,208 / 124,645,632 params unfrozen (17.5%)
  Output logits shape: torch.Size([9, 1])
  Logits: [-0.534763753414154, -0.4476749897003174, 0.2033270001411438, -0.242214173078537, 0.05039321631193161, -0.12132880836725235, 0.7779468297958374, 0.4942759573459625, 0.7595531940460205]
Smoke test passed.

Fold: test year = 2012
Dataset built: 8 cohorts
Dataset built: 1 cohorts


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa: 21,854,208 / 124,645,632 params unfrozen (17.5%)
  Epoch 01 | loss 2.2987 | train acc 0/8

Critic embedding similarities after this epoch:
  Amour                          vs Argo                           : 0.9963
  Amour                          vs Beasts of the Southern Wild    : 0.9970
  Amour                          vs Django Unchained               : 0.9966
  Amour                          vs Les MisÃ©rables                : 0.9971
  Amour                          vs Life of Pi                     : 0.9972
  Amour                          vs Lincoln                        : 0.9976
  Amour                          vs Silver Linings Playbook        : 0.9975
  Amour                          vs Zero Dark Thirty               : 0.9977
  Argo                           vs Beasts of the Southern Wild    : 0.9969
  Argo                           vs Django Unchained               : 0.9981
  Argo                           vs Les MisÃ©rables                : 0.9977
  Argo          

KeyboardInterrupt: 

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

# ── PRECOMPUTE WITH SENTENCE TRANSFORMERS ─────────────────────────────────────
def precompute_st_embeddings(nominees_df, metacritic_df, imdb_df,
                              max_critic_reviews=40, max_imdb_reviews=50,
                              save_path="/content/drive/MyDrive/oscar_st_embeddings.pt",
                              device="cuda"):
    """
    Encode all reviews using sentence-transformers all-mpnet-base-v2.
    Produces genuinely discriminative embeddings — different films
    will have meaningfully different vectors.
    Saves to Drive so you never recompute.
    """
    print("Loading sentence transformer...")
    st_model = SentenceTransformer("all-mpnet-base-v2", device=device)
    st_model.eval()

    cache = {}

    for year, year_nominees in nominees_df.groupby("ceremony_year"):
        print(f"\nEncoding year {year}...")
        films      = year_nominees["film_title"].tolist()
        winner_row = year_nominees[year_nominees["winner"] == 1]

        if winner_row.empty:
            print(f"  Warning: no winner for {year}, skipping")
            continue

        winner_title = winner_row["film_title"].iloc[0]
        if winner_title not in films:
            print(f"  Warning: winner not in films for {year}, skipping")
            continue

        winner_idx = films.index(winner_title)
        cache[year] = []

        for film in films:
            # Get critic reviews
            critic_texts = metacritic_df[
                (metacritic_df["film_title"]    == film) &
                (metacritic_df["ceremony_year"] == year)
            ]["clean_text"].dropna().tolist()[:max_critic_reviews]

            # Get IMDb reviews
            imdb_texts = imdb_df[
                (imdb_df["film_title"]    == film) &
                (imdb_df["ceremony_year"] == year)
            ]["clean_text"].dropna().tolist()[:max_imdb_reviews]

            # Fallbacks
            if not critic_texts:
                print(f"  Warning: no critic reviews for {film}")
                critic_texts = ["no critic reviews available"]
            if not imdb_texts:
                print(f"  Warning: no IMDb reviews for {film}")
                imdb_texts = ["no user reviews available"]

            # Encode — sentence transformers handle batching internally
            with torch.no_grad():
                c_emb = st_model.encode(
                    critic_texts,
                    convert_to_tensor=True,
                    show_progress_bar=False
                )  # [num_critic_reviews, 768]

                i_emb = st_model.encode(
                    imdb_texts,
                    convert_to_tensor=True,
                    show_progress_bar=False
                )  # [num_imdb_reviews, 768]

            # Pad to max_reviews so tensors are uniform shape
            c_emb = pad_embeddings(c_emb, max_critic_reviews)  # [max_critic, 768]
            i_emb = pad_embeddings(i_emb, max_imdb_reviews)    # [max_imdb,   768]

            # Padding masks — True where padded
            c_pad = torch.zeros(max_critic_reviews, dtype=torch.bool)
            i_pad = torch.zeros(max_imdb_reviews,   dtype=torch.bool)
            c_pad[len(critic_texts):] = True
            i_pad[len(imdb_texts):]   = True

            cache[year].append({
                "film":       film,
                "critic_emb": c_emb.cpu(),   # [max_critic, 768]
                "imdb_emb":   i_emb.cpu(),   # [max_imdb,   768]
                "critic_pad": c_pad,         # [max_critic]
                "imdb_pad":   i_pad,         # [max_imdb]
                "winner_idx": winner_idx,
            })

            print(f"  {film}: {len(critic_texts)} critic, {len(imdb_texts)} IMDb reviews")

    torch.save(cache, save_path)
    print(f"\nSaved to {save_path}")
    print(f"Years cached: {sorted(cache.keys())}")
    return cache


def pad_embeddings(emb_tensor, target_len):
    """
    Pad embedding tensor to target_len with zero vectors.
    emb_tensor: [n, 768]
    returns:    [target_len, 768]
    """
    n, dim = emb_tensor.shape
    if n >= target_len:
        return emb_tensor[:target_len]
    pad = torch.zeros(target_len - n, dim, device=emb_tensor.device)
    return torch.cat([emb_tensor, pad], dim=0)


# ── SIMILARITY CHECK ──────────────────────────────────────────────────────────
def check_st_similarities(cache, year=2012):
    """
    Run this immediately after precompute to verify
    embeddings are now discriminative.
    Target: similarities in 0.60-0.85 range.
    """
    cohort = cache[year]
    vecs   = [c["critic_emb"][~c["critic_pad"]].mean(dim=0) for c in cohort]

    print(f"\nSentence transformer critic similarities ({year} cohort):")
    for i in range(len(cohort)):
        for j in range(i+1, len(cohort)):
            sim = torch.nn.functional.cosine_similarity(
                vecs[i].unsqueeze(0),
                vecs[j].unsqueeze(0)
            ).item()
            print(f"  {cohort[i]['film']:35s} vs {cohort[j]['film']:35s} : {sim:.4f}")


# ── RUN PRECOMPUTE ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

cache_st = precompute_st_embeddings(
    nominees_df    = nominees_df_filtered,
    metacritic_df  = metacritic_df,
    imdb_df        = imdb_df,
    max_critic_reviews = 40,
    max_imdb_reviews   = 50,
    save_path      = "/content/drive/MyDrive/oscar_st_embeddings.pt",
    device         = device
)

# Check similarities immediately
check_st_similarities(cache_st, year=2012)

Mounted at /content/drive
Loading sentence transformer...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Encoding year 2012...
  The Artist (I): 40 critic, 50 IMDb reviews
  The Descendants: 40 critic, 50 IMDb reviews
  Extremely Loud & Incredibly Close: 40 critic, 50 IMDb reviews
  The Help: 40 critic, 50 IMDb reviews
  Hugo: 40 critic, 50 IMDb reviews
  Midnight in Paris: 40 critic, 50 IMDb reviews
  Moneyball: 40 critic, 50 IMDb reviews
  The Tree of Life: 40 critic, 50 IMDb reviews
  War Horse: 40 critic, 50 IMDb reviews

Encoding year 2013...
  Amour: 40 critic, 11 IMDb reviews
  Argo: 40 critic, 11 IMDb reviews
  Beasts of the Southern Wild: 40 critic, 11 IMDb reviews
  Django Unchained: 40 critic, 11 IMDb reviews
  Les MisÃ©rables: 40 critic, 1 IMDb reviews
  Life of Pi: 40 critic, 10 IMDb reviews
  Lincoln: 40 critic, 11 IMDb reviews
  Silver Linings Playbook: 40 critic, 16 IMDb reviews
  Zero Dark Thirty: 40 critic, 7 IMDb reviews

Encoding year 2014...
  American Hustle: 40 critic, 50 IMDb reviews
  Captain Phillips: 40 critic, 50 IMDb reviews
  Dallas Buyers Club: 40 critic, 5

In [ ]:
 # ── EVALUATION ────────────────────────────────────────────────────────────────
def evaluate_results(results, model_name="Model", k=3):
    """
    Unified evaluation for all three models.
    Computes Top-1, Top-3, and MRR with random baselines.
    Requires results dicts to have 'probs' and 'films' keys.
    """
    n       = len(results)
    n_films = np.mean([len(r["films"]) for r in results])

    top1_correct = 0
    topk_correct = 0
    rrs          = []

    print(f"\n{'='*56}")
    print(f"{model_name} — Evaluation Summary")
    print(f"{'='*56}")

    for r in results:
        winner = r["winner"]
        films  = r["films"]
        probs  = r["probs"]

        ranked      = sorted(zip(films, probs), key=lambda x: x[1], reverse=True)
        ranked_films = [f for f, p in ranked]

        # Top-1
        in_top1 = (ranked_films[0] == winner)
        top1_correct += int(in_top1)

        # Top-K
        in_topk = winner in ranked_films[:k]
        topk_correct += int(in_topk)

        # Reciprocal rank
        rank = ranked_films.index(winner) + 1
        rrs.append(1.0 / rank)

        mark1 = "✓" if in_top1 else "✗"
        markk = "✓" if in_topk else "✗"
        print(f"  {r['test_year']} | Top-1 {mark1} | Top-{k} {markk} | "
              f"RR: {1.0/rank:.3f} | "
              f"predicted: {r['predicted']:30s} | actual: {winner}")

    mrr          = np.mean(rrs)
    random_top1  = 1 / n_films
    random_topk  = k / n_films
    random_mrr   = np.mean([1/i for i in range(1, int(n_films)+1)])

    print(f"\n  Top-1 accuracy : {top1_correct}/{n} = {top1_correct/n:.1%}  "
          f"(random: {random_top1:.1%})")
    print(f"  Top-{k} accuracy : {topk_correct}/{n} = {topk_correct/n:.1%}  "
          f"(random: {random_topk:.1%})")
    print(f"  MRR            : {mrr:.4f}  "
          f"(random: {random_mrr:.4f})")

    return {
        "top1": top1_correct / n,
        "top3": topk_correct / n,
        "mrr":  mrr,
    }

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

# ── MODEL ─────────────────────────────────────────────────────────────────────
class OscarPredictorAttention(nn.Module):
    """
    Dual-stream model with multi-head self-attention and cross-attention.
    Takes precomputed sentence transformer embeddings directly.
    Reduced capacity + heavy regularization for small dataset (8 training examples).

    Pipeline per film:
      1. Sentence transformer embeddings [num_reviews, 768] passed in directly
      2. Self-attention across reviews within each stream
      3. Cross-attention between streams (critic ↔ IMDb)
      4. Mean pool reviews → one vector per stream
      5. Fusion → single film logit
    """
    def __init__(
        self,
        hidden_dim=768,
        num_heads=4,               # reduced from 8
        num_self_attn_layers=1,    # reduced from 2
        dropout=0.5                # increased from 0.3
    ):
        super(OscarPredictorAttention, self).__init__()

        self.hidden_dim = hidden_dim

        # ── Self-attention over reviews within each stream ────────────────────
        critic_self_attn_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,   # reduced from hidden_dim * 4
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.critic_self_attn = nn.TransformerEncoder(
            critic_self_attn_layer,
            num_layers=num_self_attn_layers
        )

        imdb_self_attn_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )
        self.imdb_self_attn = nn.TransformerEncoder(
            imdb_self_attn_layer,
            num_layers=num_self_attn_layers
        )

        # ── Cross-attention between streams ───────────────────────────────────
        # Q=critic, K=IMDb, V=IMDb
        self.cross_attn_critic_to_imdb = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        # Q=IMDb, K=critic, V=critic
        self.cross_attn_imdb_to_critic = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm_critic_self  = nn.LayerNorm(hidden_dim)
        self.norm_imdb_self    = nn.LayerNorm(hidden_dim)
        self.norm_critic_cross = nn.LayerNorm(hidden_dim)
        self.norm_imdb_cross   = nn.LayerNorm(hidden_dim)

        # ── Fusion ────────────────────────────────────────────────────────────
        # diff and product give explicit relationship signals between streams
        fusion_input_dim = hidden_dim * 4  # critic + imdb + diff + product

        self.scorer = nn.Sequential(
            nn.Linear(fusion_input_dim, 128),  # removed middle layer, smaller
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
            # no sigmoid — softmax applied across cohort at training time
        )

    def mean_pool_reviews(self, x):
        """Collapse [batch, num_reviews, 768] → [batch, 768]"""
        return x.mean(dim=1)

    def forward(
        self,
        critic_embs,    critic_pad_mask,
        imdb_embs,      imdb_pad_mask,
    ):
        """
        critic_embs    : [n_films, max_critic_reviews, 768]
        critic_pad_mask: [n_films, max_critic_reviews]  True = padded
        imdb_embs      : [n_films, max_imdb_reviews,   768]
        imdb_pad_mask  : [n_films, max_imdb_reviews]    True = padded
        returns        : [n_films, 1] raw logit
        """
        h_critic = critic_embs
        h_imdb   = imdb_embs

        # 1. Self-attention within each stream
        h_critic_self = self.critic_self_attn(
            h_critic, src_key_padding_mask=critic_pad_mask
        )
        h_critic = self.norm_critic_self(h_critic + h_critic_self)

        h_imdb_self = self.imdb_self_attn(
            h_imdb, src_key_padding_mask=imdb_pad_mask
        )
        h_imdb = self.norm_imdb_self(h_imdb + h_imdb_self)

        # 2. Cross-attention between streams
        h_critic_cross, _ = self.cross_attn_critic_to_imdb(
            query=h_critic, key=h_imdb, value=h_imdb,
            key_padding_mask=imdb_pad_mask
        )
        h_critic = self.norm_critic_cross(h_critic + h_critic_cross)

        h_imdb_cross, _ = self.cross_attn_imdb_to_critic(
            query=h_imdb, key=h_critic, value=h_critic,
            key_padding_mask=critic_pad_mask
        )
        h_imdb = self.norm_imdb_cross(h_imdb + h_imdb_cross)

        # 3. Pool to one vector per stream
        c_critic = self.mean_pool_reviews(h_critic)   # [n_films, 768]
        c_imdb   = self.mean_pool_reviews(h_imdb)     # [n_films, 768]

        # 4. Fusion
        diff    = c_critic - c_imdb
        product = c_critic * c_imdb
        fused   = torch.cat([c_critic, c_imdb, diff, product], dim=-1)  # [n_films, 3072]

        return self.scorer(fused)                      # [n_films, 1]


# ── TRAIN / EVAL ──────────────────────────────────────────────────────────────
def train_step_cached(model, optimizer, batch, device):
    model.train()
    optimizer.zero_grad()

    critic_embs = batch["critic_embs"].squeeze(0).to(device)
    imdb_embs   = batch["imdb_embs"].squeeze(0).to(device)
    critic_pads = batch["critic_pads"].squeeze(0).to(device)
    imdb_pads   = batch["imdb_pads"].squeeze(0).to(device)
    winner_idx  = batch["winner_idx"].squeeze().to(device)

    logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)

    loss = nn.CrossEntropyLoss()(
        logits.squeeze(-1).unsqueeze(0),
        winner_idx.unsqueeze(0)
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    probs     = torch.softmax(logits.squeeze(-1), dim=0)
    predicted = probs.argmax().item()
    correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


def eval_step_cached(model, batch, device):
    model.eval()
    with torch.no_grad():
        critic_embs = batch["critic_embs"].squeeze(0).to(device)
        imdb_embs   = batch["imdb_embs"].squeeze(0).to(device)
        critic_pads = batch["critic_pads"].squeeze(0).to(device)
        imdb_pads   = batch["imdb_pads"].squeeze(0).to(device)
        winner_idx  = batch["winner_idx"].squeeze().to(device)

        logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)

        loss = nn.CrossEntropyLoss()(
            logits.squeeze(-1).unsqueeze(0),
            winner_idx.unsqueeze(0)
        )

        probs     = torch.softmax(logits.squeeze(-1), dim=0)
        predicted = probs.argmax().item()
        correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


# ── LOOCV WITH EARLY STOPPING ─────────────────────────────────────────────────
def run_loocv_cached(cache, nominees_df,
                     num_epochs=100, lr=1e-3,
                     patience=8, device="cpu"):

    all_years = sorted(cache.keys())
    results   = []

    for test_year in all_years:
        print(f"\n{'='*56}")
        print(f"Fold: test year = {test_year}")
        print(f"{'='*56}")

        train_years = [y for y in all_years if y != test_year]

        train_dataset = OscarEmbeddingDataset(cache, train_years)
        test_dataset  = OscarEmbeddingDataset(cache, [test_year])

        train_loader  = DataLoader(train_dataset, batch_size=1, shuffle=True)
        test_loader   = DataLoader(test_dataset,  batch_size=1, shuffle=False)

        model     = OscarPredictorAttention().to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=0.1       # aggressive regularization
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs
        )

        # Early stopping state
        best_loss   = float("inf")
        no_improve  = 0
        best_epoch  = 0

        for epoch in range(num_epochs):
            losses, corrects = [], []

            for batch in train_loader:
                loss, correct, _ = train_step_cached(
                    model, optimizer, batch, device
                )
                losses.append(loss)
                corrects.append(correct)

            scheduler.step()
            mean_loss = np.mean(losses)

            # Print every 10 epochs
            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1:03d} | loss {mean_loss:.4f} | "
                      f"train acc {sum(corrects)}/{len(corrects)}")

            # Early stopping
            if mean_loss < best_loss - 1e-4:
                best_loss  = mean_loss
                no_improve = 0
                best_epoch = epoch + 1
                # Save best weights
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"  Early stopping at epoch {epoch+1} "
                          f"(best epoch {best_epoch}, loss {best_loss:.4f})")
                    break

        # Restore best weights before evaluation
        model.load_state_dict(best_state)

        # Evaluate on held-out year
        print(f"\n  Results for {test_year}:")
        for batch in test_loader:
            loss, correct, probs = eval_step_cached(model, batch, device)
            films      = batch["films"]
            winner_idx = batch["winner_idx"].squeeze().item()
            pred_idx   = probs.index(max(probs))

            for i, (film, prob) in enumerate(zip(films, probs)):
                w = " ← winner"    if i == winner_idx else ""
                p = " ← predicted" if i == pred_idx   else ""
                print(f"    {film[0]:45s} {prob:.3f}{w}{p}")

            results.append({
                "test_year": test_year,
                "correct":   correct,
                "winner":    films[winner_idx][0],
                "predicted": films[pred_idx][0],
                "probs":     probs,
                "films":     [f[0] for f in films],
            })

    # Summary
    print(f"\n{'='*56}")
    print(f"LOOCV Summary")
    print(f"{'='*56}")
    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"  {mark} {r['test_year']} | "
              f"predicted: {r['predicted']:40s} | "
              f"actual: {r['winner']}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\nOverall: {sum(r['correct'] for r in results)}/{len(results)} = {accuracy:.1%}")
    print(f"Random baseline: ~12.5%")

    return results

# ── RUN ───────────────────────────────────────────────────────────────────────
results_st = run_loocv_cached(
    cache       = cache_st,
    nominees_df = nominees_df_filtered,
    num_epochs  = 100,
    lr          = 1e-3,
    patience    = 8,
    device      = device
)
eval_attn = evaluate_results(results_st, model_name="Model 1 — Attention + Cross-Attention", k=3)


NameError: name 'cache_st' is not defined

Model 1 Evaluation: basically it's super good in training - basically memorizes the answer, then in the test year is 100% confident on something random in inference.

**Model 2 - Baseline BERT language model**

**Hypothesis:** BERT's pretrained English understanding alone,
without architectural sophistication, provides meaningful signal
above a non-contextual baseline

**Features:**

- One shared encoder: no domain separation between critic and IMDb
- Mean pooling as entire aggregation strategy: every review weighted equally, no attention
- Simple concatenation: no diff, no product, no cross-stream interaction
- One MLP head instead of 8 that learn different review aspects that relate to a Best Picture win (e.g. emotions in writing, logic, commentary related to technical aspects of film)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader


# ── MODEL ─────────────────────────────────────────────────────────────────────
class OscarPredictorSimple(nn.Module):
    """
    Model 2 — sentence transformer embeddings + simple MLP.
    No self-attention, no cross-attention.
    Tests whether attention adds value over simple mean pooling
    on the same embedding space.

    Pipeline per film:
      1. Precomputed sentence transformer embeddings passed in directly
      2. Masked mean pool critic reviews → one vector
      3. Masked mean pool IMDb reviews → one vector
      4. Concat with diff + product signals
      5. Tiny MLP → single film logit
    """
    def __init__(self, hidden_dim=768, dropout=0.5):
        super(OscarPredictorSimple, self).__init__()

        fusion_input_dim = hidden_dim * 4  # critic + imdb + diff + product

        self.scorer = nn.Sequential(
            nn.Linear(fusion_input_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def masked_mean(self, embs, pad_mask):
        """
        Mean pool embeddings ignoring padded reviews.
        embs    : [n_films, num_reviews, 768]
        pad_mask: [n_films, num_reviews]  True = padded
        returns : [n_films, 768]
        """
        real_mask = (~pad_mask).float().unsqueeze(-1)
        return (embs * real_mask).sum(dim=1) / \
               real_mask.sum(dim=1).clamp(min=1e-8)

    def forward(self, critic_embs, critic_pad_mask,
                      imdb_embs,   imdb_pad_mask):
        c_critic = self.masked_mean(critic_embs, critic_pad_mask)
        c_imdb   = self.masked_mean(imdb_embs,   imdb_pad_mask)

        diff    = c_critic - c_imdb
        product = c_critic * c_imdb

        fused = torch.cat([c_critic, c_imdb, diff, product], dim=-1)
        return self.scorer(fused)


# ── TRAIN STEP ────────────────────────────────────────────────────────────────
def train_step_simple(model, optimizer, batch, device):
    model.train()
    optimizer.zero_grad()

    critic_embs = batch["critic_embs"].squeeze(0).to(device)
    imdb_embs   = batch["imdb_embs"].squeeze(0).to(device)
    critic_pads = batch["critic_pads"].squeeze(0).to(device)
    imdb_pads   = batch["imdb_pads"].squeeze(0).to(device)
    winner_idx  = batch["winner_idx"].squeeze().to(device)

    logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)

    loss = nn.CrossEntropyLoss()(
        logits.squeeze(-1).unsqueeze(0),
        winner_idx.unsqueeze(0)
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    probs     = torch.softmax(logits.squeeze(-1), dim=0)
    predicted = probs.argmax().item()
    correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


# ── EVAL STEP ─────────────────────────────────────────────────────────────────
def eval_step_simple(model, batch, device):
    model.eval()
    with torch.no_grad():
        critic_embs = batch["critic_embs"].squeeze(0).to(device)
        imdb_embs   = batch["imdb_embs"].squeeze(0).to(device)
        critic_pads = batch["critic_pads"].squeeze(0).to(device)
        imdb_pads   = batch["imdb_pads"].squeeze(0).to(device)
        winner_idx  = batch["winner_idx"].squeeze().to(device)

        logits = model(critic_embs, critic_pads, imdb_embs, imdb_pads)

        loss = nn.CrossEntropyLoss()(
            logits.squeeze(-1).unsqueeze(0),
            winner_idx.unsqueeze(0)
        )

        probs     = torch.softmax(logits.squeeze(-1), dim=0)
        predicted = probs.argmax().item()
        correct   = (predicted == winner_idx.item())

    return loss.item(), correct, probs.detach().cpu().tolist()


# ── SUMMARIZE ─────────────────────────────────────────────────────────────────
def summarize_results(results, k=3):
    top1_correct = 0
    topk_correct = 0

    print(f"\n{'='*56}")
    print(f"Results Summary (Top-1 and Top-{k})")
    print(f"{'='*56}")

    for r in results:
        winner    = r["winner"]
        predicted = r["predicted"]
        probs     = r["probs"]
        films     = r["films"]

        ranked      = sorted(zip(films, probs), key=lambda x: x[1], reverse=True)
        top_k_films = [f for f, p in ranked[:k]]

        in_top1 = (predicted == winner)
        in_topk = (winner in top_k_films)

        top1_correct += int(in_top1)
        topk_correct += int(in_topk)

        mark1 = "✓" if in_top1 else "✗"
        markk = "✓" if in_topk else "✗"

        print(f"  {r['test_year']} | Top-1 {mark1} | Top-{k} {markk} | "
              f"predicted: {predicted:35s} | actual: {winner}")

    total = len(results)
    print(f"\nTop-1 accuracy : {top1_correct}/{total} = {top1_correct/total:.1%}")
    print(f"Top-{k} accuracy : {topk_correct}/{total} = {topk_correct/total:.1%}")
    print(f"Random Top-1   : ~12.5%")
    print(f"Random Top-{k}   : ~{k/9:.1%}")

    return top1_correct, topk_correct


# ── LOOCV ─────────────────────────────────────────────────────────────────────
def run_loocv_simple(cache, nominees_df,
                     num_epochs=100, lr=1e-3,
                     patience=8, device="cpu"):

    all_years = sorted(cache.keys())
    results   = []

    for test_year in all_years:
        print(f"\n{'='*56}")
        print(f"Fold: test year = {test_year}  [Simple MLP]")
        print(f"{'='*56}")

        train_years = [y for y in all_years if y != test_year]

        train_dataset = OscarEmbeddingDataset(cache, train_years)
        test_dataset  = OscarEmbeddingDataset(cache, [test_year])

        train_loader  = DataLoader(train_dataset, batch_size=1, shuffle=True)
        test_loader   = DataLoader(test_dataset,  batch_size=1, shuffle=False)

        model     = OscarPredictorSimple().to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=0.1
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs
        )

        best_loss  = float("inf")
        no_improve = 0
        best_epoch = 0
        best_state = None

        for epoch in range(num_epochs):
            losses, corrects = [], []

            for batch in train_loader:
                loss, correct, _ = train_step_simple(
                    model, optimizer, batch, device
                )
                losses.append(loss)
                corrects.append(correct)

            scheduler.step()
            mean_loss = np.mean(losses)

            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1:03d} | loss {mean_loss:.4f} | "
                      f"train acc {sum(corrects)}/{len(corrects)}")

            if mean_loss < best_loss - 1e-4:
                best_loss  = mean_loss
                no_improve = 0
                best_epoch = epoch + 1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"  Early stopping at epoch {epoch+1} "
                          f"(best epoch {best_epoch}, loss {best_loss:.4f})")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        print(f"\n  Results for {test_year}:")
        for batch in test_loader:
            loss, correct, probs = eval_step_simple(model, batch, device)
            films      = batch["films"]
            winner_idx = batch["winner_idx"].squeeze().item()
            pred_idx   = probs.index(max(probs))

            for i, (film, prob) in enumerate(zip(films, probs)):
                w = " ← winner"    if i == winner_idx else ""
                p = " ← predicted" if i == pred_idx   else ""
                print(f"    {film[0]:45s} {prob:.3f}{w}{p}")

            # Single correct append with all fields
            results.append({
                "test_year": test_year,
                "correct":   correct,
                "winner":    films[winner_idx][0],
                "predicted": films[pred_idx][0],
                "probs":     probs,
                "films":     [f[0] for f in films],
            })

    # Summary
    print(f"\n{'='*56}")
    print(f"Simple MLP LOOCV Summary")
    print(f"{'='*56}")
    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"  {mark} {r['test_year']} | "
              f"predicted: {r['predicted']:40s} | "
              f"actual: {r['winner']}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\nOverall: {sum(r['correct'] for r in results)}/{len(results)} = {accuracy:.1%}")
    print(f"Random baseline: ~12.5%")

    return results


# ── RUN ───────────────────────────────────────────────────────────────────────
results_simple = run_loocv_simple(
    cache       = cache_st,
    nominees_df = nominees_df_filtered,
    num_epochs  = 100,
    lr          = 1e-3,
    patience    = 8,
    device      = device
)
eval_simple = evaluate_results(results_simple, model_name="Model 2 — Simple MLP", k=3)


NameError: name 'cache_st' is not defined

### **Model 3: TF-IDF Basic Concatenation**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp


# ── TEXT PREPARATION ──────────────────────────────────────────────────────────
def prepare_texts(nominees_df, metacritic_df, imdb_df, year):
    """
    For a given year, concatenate all critic reviews and all IMDb reviews
    into one string per film. Returns list of (film, critic_text, imdb_text, is_winner).
    """
    year_nominees = nominees_df[nominees_df["ceremony_year"] == year]
    films         = year_nominees["film_title"].tolist()
    winner_row    = year_nominees[year_nominees["winner"] == 1]
    winner_title  = winner_row["film_title"].iloc[0] if not winner_row.empty else None

    film_data = []

    for film in films:
        # Concatenate all critic quotes into one string
        critic_texts = metacritic_df[
            (metacritic_df["film_title"]    == film) &
            (metacritic_df["ceremony_year"] == year)
        ]["clean_text"].dropna().tolist()

        # Concatenate all IMDb reviews into one string
        imdb_texts = imdb_df[
            (imdb_df["film_title"]    == film) &
            (imdb_df["ceremony_year"] == year)
        ]["clean_text"].dropna().tolist()

        critic_combined = " ".join(critic_texts) if critic_texts else "no reviews"
        imdb_combined   = " ".join(imdb_texts)   if imdb_texts   else "no reviews"
        is_winner       = 1 if film == winner_title else 0

        film_data.append({
            "film":          film,
            "critic_text":   critic_combined,
            "imdb_text":     imdb_combined,
            "is_winner":     is_winner,
        })

    return film_data


# ── TFIDF MODEL ───────────────────────────────────────────────────────────────
class OscarPredictorTFIDF:
    """
    Model 3 — TF-IDF + Logistic Regression.
    Non-neural baseline. No embeddings, no training loop, no GPU.

    Critic and IMDb reviews are each vectorized with separate TF-IDF
    vectorizers then concatenated into one feature vector per film.
    Logistic regression predicts P(winner).

    Purpose: sanity check baseline.
    If neural models don't beat this, something is wrong with the
    neural pipeline or the data is too noisy for any model.
    """
    def __init__(self, max_features=5000, C=0.01):
        # Separate vectorizers — critic and audience use different vocabulary
        # min_df=1 because corpus is tiny (77 films total)
        self.critic_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),      # unigrams + bigrams
            sublinear_tf=True,       # log normalization on term frequency
            strip_accents="unicode",
            min_df=1,
        )
        self.imdb_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
            min_df=1,
        )
        # C=0.01 — heavy regularization essential for small dataset
        # class_weight="balanced" — corrects 1 winner vs 8 nominees imbalance
        self.classifier = LogisticRegression(
            C=C,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )

    def _build_features(self, film_data_list, fit=False):
        """
        Vectorize critic and IMDb texts and concatenate.
        fit=True during training, fit=False during test (transform only).

        returns: sparse matrix [n_films, max_features * 2]
        """
        critic_texts = [d["critic_text"] for d in film_data_list]
        imdb_texts   = [d["imdb_text"]   for d in film_data_list]

        if fit:
            critic_features = self.critic_vectorizer.fit_transform(critic_texts)
            imdb_features   = self.imdb_vectorizer.fit_transform(imdb_texts)
        else:
            critic_features = self.critic_vectorizer.transform(critic_texts)
            imdb_features   = self.imdb_vectorizer.transform(imdb_texts)

        # Horizontally stack critic and IMDb features
        return sp.hstack([critic_features, imdb_features])

    def fit(self, film_data_list):
        """Train on a list of film dicts (from multiple years)."""
        X = self._build_features(film_data_list, fit=True)
        y = np.array([d["is_winner"] for d in film_data_list])
        self.classifier.fit(X, y)

    def predict_proba(self, film_data_list):
        """
        Returns P(winner) for each film.
        Normalized to sum to 1 across cohort for comparability
        with softmax outputs from neural models.
        """
        X     = self._build_features(film_data_list, fit=False)
        probs = self.classifier.predict_proba(X)[:, 1]  # P(winner)

        # Normalize across cohort — sum to 1
        total = probs.sum()
        if total > 0:
            probs = probs / total

        return probs


# ── LOOCV ─────────────────────────────────────────────────────────────────────
def run_loocv_tfidf(nominees_df, metacritic_df, imdb_df,
                    max_features=5000, C=0.01):
    """
    Leave-one-year-out cross validation for TF-IDF model.
    Fits vectorizer and classifier on training years only.
    Transforms test year using training vocabulary — no leakage.
    """
    all_years = sorted(nominees_df["ceremony_year"].unique())
    results   = []

    for test_year in all_years:
        print(f"\n{'='*56}")
        print(f"Fold: test year = {test_year}  [TF-IDF]")
        print(f"{'='*56}")

        train_years = [y for y in all_years if y != test_year]

        # Build training data — all films from all training years
        train_data = []
        for year in train_years:
            train_data.extend(prepare_texts(nominees_df, metacritic_df, imdb_df, year))

        # Build test data — films from held-out year only
        test_data = prepare_texts(nominees_df, metacritic_df, imdb_df, test_year)

        print(f"  Train: {len(train_data)} films ({sum(d['is_winner'] for d in train_data)} winners)")
        print(f"  Test:  {len(test_data)} films")

        # Fit on training data, transform test data
        model = OscarPredictorTFIDF(max_features=max_features, C=C)
        model.fit(train_data)
        probs = model.predict_proba(test_data)

        # Find winner and prediction
        true_idx = next(i for i, d in enumerate(test_data) if d["is_winner"] == 1)
        pred_idx = probs.argmax()
        correct  = (pred_idx == true_idx)
        films    = [d["film"] for d in test_data]

        # Print results
        print(f"\n  Results for {test_year}:")
        for i, (film, prob) in enumerate(zip(films, probs)):
            w = " ← winner"    if i == true_idx else ""
            p = " ← predicted" if i == pred_idx else ""
            print(f"    {film:45s} {prob:.3f}{w}{p}")

        results.append({
            "test_year": test_year,
            "correct":   correct,
            "winner":    films[true_idx],
            "predicted": films[pred_idx],
            "probs":     probs.tolist(),
            "films":     films,
        })

    # Summary
    print(f"\n{'='*56}")
    print(f"TF-IDF LOOCV Summary")
    print(f"{'='*56}")
    for r in results:
        mark = "✓" if r["correct"] else "✗"
        print(f"  {mark} {r['test_year']} | "
              f"predicted: {r['predicted']:40s} | "
              f"actual: {r['winner']}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\nOverall: {sum(r['correct'] for r in results)}/{len(results)} = {accuracy:.1%}")
    print(f"Random baseline: ~12.5%")

    return results

In [ ]:
# ------------ C = 20 ------------
results_tfidf = run_loocv_tfidf(
    nominees_df   = nominees_df_filtered,
    metacritic_df = metacritic_df,
    imdb_df       = imdb_df,
    max_features  = 5000,
    C             = 20
)
eval_tfidf = evaluate_results(results_tfidf, model_name="Model 3 — TF-IDF", k=3)

In [ ]:
# ------------ C = 1 OR 10 ------------
for C in [1.0, 10.0]:
    print(f"\n{'#'*56}")
    print(f"# C = {C}")
    print(f"{'#'*56}")
    results_tfidf = run_loocv_tfidf(
        nominees_df   = nominees_df_filtered,
        metacritic_df = metacritic_df,
        imdb_df       = imdb_df,
        max_features  = 5000,
        C             = C
    )
    summarize_results_tfidf(results_tfidf, k=3)


########################################################
# C = 1.0
########################################################

Fold: test year = 2012  [TF-IDF]
  Train: 69 films (8 winners)
  Test:  9 films

  Results for 2012:
    The Artist (I)                                0.113 ← winner
    The Descendants                               0.104
    Extremely Loud & Incredibly Close             0.110
    The Help                                      0.124 ← predicted
    Hugo                                          0.113
    Midnight in Paris                             0.110
    Moneyball                                     0.109
    The Tree of Life                              0.113
    War Horse                                     0.104

Fold: test year = 2013  [TF-IDF]
  Train: 69 films (8 winners)
  Test:  9 films

  Results for 2013:
    Amour                                         0.118 ← predicted
    Argo                                          0.113 ← winner
    Beasts of

In [ ]:
# ------------ C = 20 ------------
for C in [20.0]:
    print(f"\n{'#'*56}")
    print(f"# C = {C}")
    print(f"{'#'*56}")
    results_tfidf = run_loocv_tfidf(
        nominees_df   = nominees_df_filtered,
        metacritic_df = metacritic_df,
        imdb_df       = imdb_df,
        max_features  = 5000,
        C             = C
    )
    summarize_results_tfidf(results_tfidf, k=3)


########################################################
# C = 20
########################################################

Fold: test year = 2012  [TF-IDF]
  Train: 69 films (8 winners)
  Test:  9 films

  Results for 2012:
    The Artist (I)                                0.116 ← winner
    The Descendants                               0.075
    Extremely Loud & Incredibly Close             0.106
    The Help                                      0.180 ← predicted
    Hugo                                          0.120
    Midnight in Paris                             0.106
    Moneyball                                     0.099
    The Tree of Life                              0.119
    War Horse                                     0.080

Fold: test year = 2013  [TF-IDF]
  Train: 69 films (8 winners)
  Test:  9 films

  Results for 2013:
    Amour                                         0.145 ← predicted
    Argo                                          0.123 ← winner
    Beasts of 

In [ ]:
# ------------ C = 50 ------------
for C in [50.0]:
    print(f"\n{'#'*56}")
    print(f"# C = {C}")
    print(f"{'#'*56}")
    results_tfidf = run_loocv_tfidf(
        nominees_df   = nominees_df_filtered,
        metacritic_df = metacritic_df,
        imdb_df       = imdb_df,
        max_features  = 5000,
        C             = C
    )
    summarize_results_tfidf(results_tfidf, k=3)


########################################################
# C = 50.0
########################################################

Fold: test year = 2012  [TF-IDF]
  Train: 69 films (8 winners)
  Test:  9 films

  Results for 2012:
    The Artist (I)                                0.116 ← winner
    The Descendants                               0.065
    Extremely Loud & Incredibly Close             0.103
    The Help                                      0.207 ← predicted
    Hugo                                          0.122
    Midnight in Paris                             0.102
    Moneyball                                     0.094
    The Tree of Life                              0.120
    War Horse                                     0.071

Fold: test year = 2013  [TF-IDF]
  Train: 69 films (8 winners)
  Test:  9 films

  Results for 2013:
    Amour                                         0.161 ← predicted
    Argo                                          0.128 ← winner
    Beasts o

In [ ]:
# ------------ C = 100 ------------
for C in [100.0]:
    print(f"\n{'#'*56}")
    print(f"# C = {C}")
    print(f"{'#'*56}")
    results_tfidf = run_loocv_tfidf(
        nominees_df   = nominees_df_filtered,
        metacritic_df = metacritic_df,
        imdb_df       = imdb_df,
        max_features  = 5000,
        C             = C
    )
    summarize_results_tfidf(results_tfidf, k=3)


########################################################
# C = 100.0
########################################################

Fold: test year = 2012  [TF-IDF]
  Train: 69 films (8 winners)
  Test:  9 films

  Results for 2012:
    The Artist (I)                                0.115 ← winner
    The Descendants                               0.059
    Extremely Loud & Incredibly Close             0.100
    The Help                                      0.231 ← predicted
    Hugo                                          0.121
    Midnight in Paris                             0.101
    Moneyball                                     0.089
    The Tree of Life                              0.121
    War Horse                                     0.064

Fold: test year = 2013  [TF-IDF]
  Train: 69 films (8 winners)
  Test:  9 films

  Results for 2013:
    Amour                                         0.168 ← predicted
    Argo                                          0.131 ← winner
    Beasts 

In [ ]:
# ------------ C = 250 ------------
for C in [250.0]:
    print(f"\n{'#'*56}")
    print(f"# C = {C}")
    print(f"{'#'*56}")
    results_tfidf = run_loocv_tfidf(
        nominees_df   = nominees_df_filtered,
        metacritic_df = metacritic_df,
        imdb_df       = imdb_df,
        max_features  = 5000,
        C             = C
    )
    summarize_results_tfidf(results_tfidf, k=3)

## Evaluation (NEEDS TO BE IMPLEMENTED)
Methods:
  - Accuracy
  - Top-1, Top-5
  - Others?